In [ ]:
# Cell 1: Environment Setup and Imports

import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
from tqdm.notebook import tqdm
import lightgbm as lgb
import warnings

# Core ML Dependencies
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import timm
from sentence_transformers import SentenceTransformer
from src.utils import download_images  # Ensure utils.py exists under src/ directory

# Suppress unnecessary warnings
warnings.filterwarnings('ignore')

# Define base data directory
BASE_PATH = Path('./data')

# --- UPDATED IMAGE DIRECTORY SETTINGS ---
# Adjusted to align correctly with your file structure.
# TRAIN_DIR now corresponds to the directory containing training images.
# TEST_DIR points to the directory with testing images.

TRAIN_DIR = BASE_PATH / 'test_images'   # Directory with training set images
TEST_DIR = BASE_PATH / 'train_images'   # Directory with testing set images

# Create directories if they don't already exist
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

# Select computation device (GPU/Metal/CPU)
if torch.cuda.is_available():
    COMPUTE_DEVICE = "cuda"
elif torch.backends.mps.is_available():
    COMPUTE_DEVICE = "mps"
else:
    COMPUTE_DEVICE = "cpu"

print(f"Initialization done. Running on device: {COMPUTE_DEVICE}")
print("\n--- Directory Configuration ---")
print(f"TRAIN_DIR path: {TRAIN_DIR}")
print(f"TEST_DIR path:  {TEST_DIR}")


Setup Complete. Using device: mps

--- Path Correction ---
TRAIN_IMAGE_DIR is now set to: data/test_images
TEST_IMAGE_DIR is now set to:  data/train_images


In [2]:
# Load Data

df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_test = pd.read_csv(DATA_DIR / 'test.csv')

print(f"Train data shape: {df_train.shape}")
print(f"Test data shape: {df_test.shape}")

print("\nTrain data sample:")
df_train.head()

Train data shape: (75000, 4)
Test data shape: (75000, 3)

Train data sample:


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [ ]:
# Cell 4 (UPDATED): Assign correct image directory paths

# Specify folder locations consistent with your dataset structure
TRAIN_FOLDER = BASE_PATH / 'train_images'
TEST_FOLDER = BASE_PATH / 'test_images'

# Validate that these directories exist (create them if missing)
TRAIN_FOLDER.mkdir(parents=True, exist_ok=True)
TEST_FOLDER.mkdir(parents=True, exist_ok=True)

print("Directory configuration verified successfully:")
print(f"Training image directory: {TRAIN_FOLDER}")
print(f"Testing image directory:  {TEST_FOLDER}")


Paths are now correctly set:
Train images path: data/train_images
Test images path: data/test_images


In [ ]:
# Cell 5 (UPDATED): Download training images with basic error handling

# import requests
# from PIL import Image
# from io import BytesIO
# from tqdm import tqdm
# import os

# def fetch_image(entry, save_dir):
#     """Retrieve a single image using its sample_id and URL."""
#     img_id = entry['sample_id']
#     img_link = entry['image_link']
#     img_file = os.path.join(save_dir, f"{img_id}.jpg")
    
#     # Skip if already present
#     if os.path.exists(img_file):
#         return True
    
#     try:
#         response = requests.get(img_link, timeout=10)
#         if response.status_code == 200:
#             picture = Image.open(BytesIO(response.content)).convert('RGB')
#             picture.save(img_file, 'JPEG')
#             return True
#     except Exception:
#         return False
    
#     return False

# print("Fetching training dataset images...")
# print("This may take a while — progress bar will display status.")

# success_count = 0
# failure_count = 0

# for index, entry in tqdm(df_train.iterrows(), total=len(df_train), desc="Training images"):
#     if fetch_image(entry, str(TRAIN_FOLDER)):
#         success_count += 1
#     else:
#         failure_count += 1

#     # Add a short delay every 100 images to prevent throttling
#     if index % 100 == 0 and index > 0:
#         time.sleep(0.5)

# print("\nTraining image download finished!")
# print(f"Successful: {success_count}, Failed: {failure_count}")

print(f"Total images available in 'train_images': {len(list(TRAIN_FOLDER.glob('*.jpg')))}")


# Cell 6 (UPDATED): Download test images (if required)

# print("Fetching test dataset images...")

# success_count = 0
# failure_count = 0

# for index, entry in tqdm(df_test.iterrows(), total=len(df_test), desc="Test images"):
#     if fetch_image(entry, str(TEST_FOLDER)):
#         success_count += 1
#     else:
#         failure_count += 1

#     if index % 100 == 0 and index > 0:
#         time.sleep(0.5)

# print("\nTest image download finished!")
# print(f"Successful: {success_count}, Failed: {failure_count}")

print(f"Total images available in 'test_images': {len(list(TEST_FOLDER.glob('*.jpg')))}")


Total files in train_images: 74999
Total files in test_images: 74999


In [ ]:
# Remove incorrect or extra images from test directory

from tqdm import tqdm

print(f"Starting cleanup for test image directory: {TEST_FOLDER}")

# Build a set of all valid test sample IDs for quick membership checking
valid_test_ids = set(df_test['sample_id'].astype(str))

extra_files = []
# Identify image files that should not exist in the test folder
for img_path in TEST_FOLDER.glob('*.jpg'):
    if img_path.stem not in valid_test_ids:
        extra_files.append(img_path)

# If no invalid images are found
if not extra_files:
    print("No unnecessary files detected. Test folder is already clean.")
else:
    print(f"Detected {len(extra_files)} files that don't belong. Removing them...")
    for file_path in tqdm(extra_files, desc="Removing extra images"):
        file_path.unlink()  # Permanently deletes the file
    print("Cleanup completed successfully.")

# --- Verify image counts after cleanup ---
train_img_count = len(list(TRAIN_FOLDER.glob('*.jpg')))
test_img_count = len(list(TEST_FOLDER.glob('*.jpg')))

print("\n--- Image Count Summary ---")
print(f"Images in training directory ({TRAIN_FOLDER.name}): {train_img_count}")
print(f"Images in testing directory ({TEST_FOLDER.name}):  {test_img_count}")


Cleaning the test images folder: data/train_images
No extra files found. Folder is already clean!

--- Final File Counts ---
Images in TRAIN folder (test_images): 74999
Images in TEST folder (train_images):  74999


In [ ]:
# Optimized and selective image downloader

# Import the main downloading utility
from src.utils import download_images
from pathlib import Path

def smart_image_fetch(dataframe, target_dir):
    """
    Scans for missing images and downloads only the absent ones.
    """
    target_path = Path(target_dir)
    print(f"--- Checking availability in '{target_path.name}' directory ---")

    # Collect IDs of expected and existing images
    required_ids = set(dataframe['sample_id'].astype(str))
    present_ids = {file.stem for file in target_path.glob('*.jpg')}
    missing = required_ids - present_ids

    print(f"Located {len(present_ids)} of {len(required_ids)} total images.")

    if not missing:
        print("All files accounted for — no download required.")
        return

    print(f"{len(missing)} image(s) missing. Beginning targeted download...")

    # Retrieve only the rows corresponding to missing images
    df_to_fetch = dataframe[dataframe['sample_id'].astype(str).isin(missing)]

    # Prepare a list of (sample_id, image_link) pairs for downloading
    download_jobs = list(zip(df_to_fetch['sample_id'], df_to_fetch['image_link']))

    # Trigger the downloader function from utils.py
    download_images(download_jobs, str(target_path))

    print("\nDownload process finished.")

# --- Execute selective downloading for training and testing datasets ---
# Ensure TRAIN_FOLDER and TEST_FOLDER are defined in setup cells
smart_image_fetch(df_train, TRAIN_FOLDER)
print("-" * 30)
smart_image_fetch(df_test, TEST_FOLDER)


--- Verifying images in 'test_images' ---
Found 74999 of 75000 expected images.
1 missing image(s) detected. Preparing to download.



Download attempt complete.
------------------------------
--- Verifying images in 'train_images' ---
Found 74999 of 75000 expected images.
1 missing image(s) detected. Preparing to download.



Download attempt complete.


In [ ]:
# Cell 7: Extract and clean catalog content

import re

def extract_features(raw_content):
    """
    Converts raw catalog_content text into structured features:
    - clean_text: combined item name, bullet points, and product description
    - quantity: numeric value (default 1.0)
    - unit: measurement unit (default 'Unknown')
    """
    if not isinstance(raw_content, str):
        raw_content = ""
        
    lines = raw_content.strip().split('\n')
    
    # Initialize default values
    product_name = ""
    bullets = []
    description = ""
    quantity = 1.0
    unit = "Unknown"

    for line in lines:
        line_lower = line.lower()
        if line_lower.startswith("item name:"):
            product_name = line[len("item name:"):].strip()
        elif line_lower.startswith("bullet point"):
            bp_text = re.sub(r'Bullet Point \d+:', '', line, flags=re.IGNORECASE).strip()
            bullets.append(bp_text)
        elif line_lower.startswith("product description:"):
            description = line[len("product description:"):].strip()
        elif line_lower.startswith("value:"):
            try:
                quantity = float(line[len("value:"):].strip())
            except (ValueError, TypeError):
                quantity = 1.0  # fallback default
        elif line_lower.startswith("unit:"):
            unit = line[len("unit:"):].strip()
            
    # Merge all text segments into a single clean_text feature
    clean_text = " ".join([product_name] + bullets + [description]).strip()
    
    return pd.Series([clean_text, quantity, unit], index=['clean_text', 'quantity', 'unit'])


# --- Apply parsing to train and test datasets ---
print("Parsing training dataset catalog content...")
train_features = df_train['catalog_content'].apply(extract_features)
df_train = pd.concat([df_train, train_features], axis=1)

print("Parsing test dataset catalog content...")
test_features = df_test['catalog_content'].apply(extract_features)
df_test = pd.concat([df_test, test_features], axis=1)

# Verify newly created columns
print("\nFeature extraction completed successfully!")
df_train[['clean_text', 'quantity', 'unit']].head()


Parsing training data...
Parsing test data...

New features created successfully!


,clean_text,quantity,unit
0,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",72.00,Fl Oz
1,"Salerno Cookies, The Original Butter Cookies, ...",32.00,Ounce
2,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",11.40,Ounce
3,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,11.25,Ounce
4,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",12.00,Count


In [8]:
# Cell 8: Generate and Save Text Embeddings

# import torch
# import numpy as np
# from tqdm import tqdm
# from pathlib import Path
# from sentence_transformers import SentenceTransformer

# # --- Device setup ---
# DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
# print(f"Using device: {DEVICE}")

# # --- Parameters ---
# TEXT_BATCH_SIZE = 33
# SAVE_DIR = Path("embeddings")
# SAVE_DIR.mkdir(exist_ok=True, parents=True)

# # --- Text Embedding Generation ---
# print("Loading text model...")
# text_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

# def generate_text_embeddings(df, column, prefix="train"):
#     """Generates and saves text embeddings in chunks."""
#     print(f"\n--- Generating Text Embeddings for '{prefix}' set ---")
#     texts = df[column].tolist()
#     EMB_CHUNK = 15000  # Process 10,000 texts at a time

#     for start in range(0, len(texts), EMB_CHUNK):
#         end = min(start + EMB_CHUNK, len(texts))
#         print(f"Processing samples {start} to {end}...")
#         batch_texts = texts[start:end]
        
#         embeds = text_model.encode(
#             batch_texts, 
#             batch_size=TEXT_BATCH_SIZE, 
#             show_progress_bar=True, 
#             convert_to_numpy=True
#         )
        
#         np.save(SAVE_DIR / f"{prefix}_text_embeds_{start}_{end}.npy", embeds)
#         print(f"Saved chunk: {prefix}_text_embeds_{start}_{end}.npy")
#         if DEVICE == "mps":
#             torch.mps.empty_cache()

# # --- Execute Text Embedding Generation ---
# #generate_text_embeddings(df_train, "clean_text", prefix="train")
# #generate_text_embeddings(df_test, "clean_text", prefix="test")

# # --- IMPORTANT: Clear model from memory ---
# #del text_model
# #if DEVICE == "mps":
 #    torch.mps.empty_cache()
print("\nText model cleared from memory.")


Text model cleared from memory.


In [9]:
# import ssl
# import torch
# import numpy as np
# from tqdm import tqdm
# from pathlib import Path
# import clip
# from PIL import Image

# # Cell 9 (UPDATED): More Memory-Efficient Image Embeddings

# import gc # Import the garbage collector

# # --- SSL Fix (just in case) ---
# ssl._create_default_https_context = ssl._create_unverified_context

# # --- Device setup ---
# DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
# print(f"Using device: {DEVICE}")

# # --- Parameters ---
# # REDUCED BATCH SIZE to lower memory usage
# IMAGE_BATCH_SIZE = 16 
# SAVE_DIR = Path("embeddings")

# # --- Image Embedding Generation ---
# print("Loading CLIP model...")
# # This is the line that was causing the crash
# clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE) 
# print("CLIP model loaded successfully.")

# def generate_image_embeddings(df, image_dir, prefix="train"):
#     """Generates and saves image embeddings in chunks with aggressive memory cleaning."""
#     print(f"\n--- Generating Image Embeddings for '{prefix}' set ---")
#     image_paths = [Path(image_dir) / f"{sid}.jpg" for sid in df['sample_id'].astype(str)]
#     EMB_CHUNK = 5000  # Process 5,000 images at a time

#     for start in range(0, len(image_paths), EMB_CHUNK):
#         end = min(start + EMB_CHUNK, len(image_paths))
#         print(f"Processing samples {start} to {end}...")
#         batch_paths = image_paths[start:end]
        
#         # Inner loop to process smaller batches
#         chunk_embeds = []
#         for i in tqdm(range(0, len(batch_paths), IMAGE_BATCH_SIZE), desc="Image batches"):
#             inner_batch_paths = batch_paths[i:i+IMAGE_BATCH_SIZE]
#             images = []
#             for p in inner_batch_paths:
#                 try:
#                     img = Image.open(p).convert("RGB")
#                     images.append(clip_preprocess(img))
#                 except (FileNotFoundError, OSError):
#                     images.append(torch.zeros(3, 224, 224)) # Placeholder for missing images
            
#             image_batch = torch.stack(images).to(DEVICE)

#             with torch.no_grad():
#                 embeds = clip_model.encode_image(image_batch).cpu().numpy()
#                 chunk_embeds.append(embeds)

#             # Forcefully clear memory
#             del image_batch
#             del images
#             if DEVICE == "mps":
#                 torch.mps.empty_cache()
#             gc.collect()

#         # Save the collected embeddings for the entire chunk
#         chunk_embeds_full = np.vstack(chunk_embeds)
#         np.save(SAVE_DIR / f"{prefix}_image_embeds_{start}_{end}.npy", chunk_embeds_full)
#         print(f"Saved chunk: {prefix}_image_embeds_{start}_{end}.npy")

# # --- Execute Image Embedding Generation ---
# generate_image_embeddings(df_train, TRAIN_IMAGE_DIR, prefix="train")

# # --- Final Cleanup ---
# del clip_model
# gc.collect()
# if DEVICE == "mps":
#     torch.mps.empty_cache()

# print("\nAll image embeddings have been generated and saved.")


In [10]:
# import ssl
# import torch
# import numpy as np
# from tqdm import tqdm
# from pathlib import Path
# import clip
# from PIL import Image
# import gc
# import pandas as pd

# # --- This script does ONLY ONE THING: generates test image embeddings ---

# print("--- Starting Test Image Embedding Generation ---")

# # --- Setup Paths ---
# DATA_DIR = Path('./data')
# TEST_IMAGE_DIR = DATA_DIR / 'train_images' # As per our corrected path setup
# SAVE_DIR = Path("embeddings")
# SAVE_DIR.mkdir(exist_ok=True, parents=True)

# # --- SSL Fix ---
# ssl._create_default_https_context = ssl._create_unverified_context

# # --- Device setup ---
# DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
# print(f"Using device: {DEVICE}")

# # --- Parameters ---
# IMAGE_BATCH_SIZE = 16 
# EMB_CHUNK = 5000

# # --- Load ONLY the necessary data ---
# print("Loading test.csv...")
# df_test = pd.read_csv(DATA_DIR / 'test.csv')

# # --- Image Embedding Generation Function ---
# def generate_image_embeddings(df, image_dir, prefix="test"):
#     print(f"\n--- Generating Image Embeddings for '{prefix}' set ---")
#     image_paths = [Path(image_dir) / f"{sid}.jpg" for sid in df['sample_id'].astype(str)]

#     for start in range(0, len(image_paths), EMB_CHUNK):
#         end = min(start + EMB_CHUNK, len(image_paths))
#         print(f"Processing samples {start} to {end}...")
#         batch_paths = image_paths[start:end]
        
#         chunk_embeds = []
#         for i in tqdm(range(0, len(batch_paths), IMAGE_BATCH_SIZE), desc="Image batches"):
#             inner_batch_paths = batch_paths[i:i+IMAGE_BATCH_SIZE]
#             images = []
#             for p in inner_batch_paths:
#                 try:
#                     img = Image.open(p).convert("RGB")
#                     images.append(clip_preprocess(img))
#                 except (FileNotFoundError, OSError):
#                     images.append(torch.zeros(3, 224, 224))
            
#             image_batch = torch.stack(images).to(DEVICE)

#             with torch.no_grad():
#                 embeds = clip_model.encode_image(image_batch).cpu().numpy()
#                 chunk_embeds.append(embeds)

#             # Forcefully clear memory
#             del image_batch, images
#             if DEVICE == "mps":
#                 torch.mps.empty_cache()
#             gc.collect()

#         chunk_embeds_full = np.vstack(chunk_embeds)
#         np.save(SAVE_DIR / f"{prefix}_image_embeds_{start}_{end}.npy", chunk_embeds_full)
#         print(f"Saved chunk: {prefix}_image_embeds_{start}_{end}.npy")

# # --- Main Execution ---
# if __name__ == "__main__":
#     print("\nLoading CLIP model...")
#     clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE) 
#     print("CLIP model loaded successfully.")

#     # Execute ONLY for the TEST set
#     #generate_image_embeddings(df_test, TEST_IMAGE_DIR, prefix="test")

#     print("\nAll test image embeddings have been generated and saved!")

In [11]:
# # Cell: Generate LAST CHUNK of TEST IMAGE Embeddings

# import ssl
# import torch
# import numpy as np
# from tqdm import tqdm
# from pathlib import Path
# import clip
# from PIL import Image
# import gc

# # --- SSL Fix ---
# ssl._create_default_https_context = ssl._create_unverified_context

# # --- Device setup ---
# DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
# print(f"Using device: {DEVICE}")

# # --- Parameters ---
# IMAGE_BATCH_SIZE = 16 
# EMB_CHUNK = 5000
# SAVE_DIR = Path("embeddings")

# # --- Image Embedding Generation ---
# print("Loading CLIP model...")
# clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE) 
# print("CLIP model loaded successfully.")

# def generate_image_embeddings(df, image_dir, prefix="test", start_from=0):
#     print(f"\n--- Generating Image Embeddings for '{prefix}' set, starting from index {start_from} ---")
#     image_paths = [Path(image_dir) / f"{sid}.jpg" for sid in df['sample_id'].astype(str)]

#     # This loop will now start from 70000
#     for start in range(start_from, len(image_paths), EMB_CHUNK):
#         end = min(start + EMB_CHUNK, len(image_paths))
#         print(f"Processing samples {start} to {end}...")
#         batch_paths = image_paths[start:end]
        
#         chunk_embeds = []
#         for i in tqdm(range(0, len(batch_paths), IMAGE_BATCH_SIZE), desc="Image batches"):
#             inner_batch_paths = batch_paths[i:i+IMAGE_BATCH_SIZE]
#             images = []
#             for p in inner_batch_paths:
#                 try:
#                     img = Image.open(p).convert("RGB")
#                     images.append(clip_preprocess(img))
#                 except (FileNotFoundError, OSError):
#                     images.append(torch.zeros(3, 224, 224))
            
#             image_batch = torch.stack(images).to(DEVICE)
#             with torch.no_grad():
#                 embeds = clip_model.encode_image(image_batch).cpu().numpy()
#                 chunk_embeds.append(embeds)

#             del image_batch, images
#             if DEVICE == "mps": torch.mps.empty_cache()
#             gc.collect()

#         chunk_embeds_full = np.vstack(chunk_embeds)
#         np.save(SAVE_DIR / f"{prefix}_image_embeds_{start}_{end}.npy", chunk_embeds_full)
#         print(f"Saved chunk: {prefix}_image_embeds_{start}_{end}.npy")

# # --- Execute for ONLY the last chunk of the TEST set ---
# generate_image_embeddings(df_test, TEST_IMAGE_DIR, prefix="test", start_from=70000)

# del clip_model
# gc.collect()
# if DEVICE == "mps":
#     torch.mps.empty_cache()

# print("\n✅ Final image chunk generated.")

In [ ]:
# Cell 10: Assemble and Save Final Feature Matrices

import numpy as np
import pandas as pd
from pathlib import Path
import gc

EMBEDDINGS_DIR = Path("embeddings")

# --- Prepare categorical 'unit' features for consistency across train/test ---
print("Generating one-hot encoded 'unit' features...")
train_unit_dummies = pd.get_dummies(df_train['unit'], prefix='unit')
test_unit_dummies = pd.get_dummies(df_test['unit'], prefix='unit')

# Align columns so train and test have the same unit feature columns
train_units_aligned, test_units_aligned = train_unit_dummies.align(test_unit_dummies, join='outer', axis=1, fill_value=0)
print("Categorical 'unit' features ready.")

def assemble_and_save(split_name, dataframe, unit_features):
    """
    Loads text and image embeddings, combines with numeric/categorical features,
    and saves the final feature matrix for a dataset split.
    """
    print(f"\n--- Processing '{split_name}' dataset ---")
    
    text_files = sorted(EMBEDDINGS_DIR.glob(f"{split_name}_text_embeds_*.npy"))
    image_files = sorted(EMBEDDINGS_DIR.glob(f"{split_name}_image_embeds_*.npy"))
    
    if not text_files or not image_files:
        raise FileNotFoundError(f"Missing embedding files for '{split_name}'. Check '{EMBEDDINGS_DIR}' folder.")

    print(f"Loading and stacking {len(text_files)} text embedding chunks...")
    text_embeddings = np.vstack([np.load(f) for f in text_files])
    
    print(f"Loading and stacking {len(image_files)} image embedding chunks...")
    image_embeddings = np.vstack([np.load(f) for f in image_files])

    print("Combining all features horizontally...")
    final_features = np.hstack([
        text_embeddings,
        image_embeddings,
        dataframe['quantity'].values.reshape(-1, 1),
        unit_features.values
    ])
    
    save_path = EMBEDDINGS_DIR / f"final_X_{split_name}.npy"
    np.save(save_path, final_features)
    print(f"Saved final feature array: {save_path} | Shape: {final_features.shape}")
    
    # Free memory
    del text_embeddings, image_embeddings, final_features
    gc.collect()

# --- Run for train and test splits ---
assemble_and_save("train", df_train, train_units_aligned)
assemble_and_save("test", df_test, test_units_aligned)

print("\nAll final feature matrices have been successfully created.")


Preparing categorical 'unit' features...
Categorical features prepared.

--- Processing 'train' set ---
Loading and combining 5 text chunks...
Loading and combining 15 image chunks...
Horizontally stacking all features...
Saved final feature array to: embeddings/final_X_train.npy with shape (75000, 1032)

--- Processing 'test' set ---
Loading and combining 5 text chunks...
Loading and combining 15 image chunks...
Horizontally stacking all features...
Saved final feature array to: embeddings/final_X_test.npy with shape (75000, 1032)

All final feature arrays have been created.


In [ ]:
# Cell 11: Load Final Features, Train LightGBM, and Generate Submission

import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
import gc

EMBEDDINGS_DIR = Path("embeddings")

# --- Load pre-assembled feature arrays ---
print("Loading final train and test feature matrices...")
X_train_final = np.load(EMBEDDINGS_DIR / "final_X_train.npy", allow_pickle=True)
X_test_final = np.load(EMBEDDINGS_DIR / "final_X_test.npy", allow_pickle=True)

# Prepare target variable (log-transformed to stabilize variance)
y_train_target = np.log1p(df_train['price'])

print("Feature arrays loaded successfully!")
print(f"X_train shape: {X_train_final.shape}")
print(f"X_test shape: {X_test_final.shape}")

# --- Train LightGBM Regressor ---
print("\nTraining LightGBM model...")
lgb_model = lgb.LGBMRegressor(
    objective='regression_l1',
    metric='rmse',
    n_estimators=2000,
    learning_rate=0.01,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    n_jobs=-1,
    seed=42,
    verbose=-1,
)

lgb_model.fit(X_train_final, y_train_target)
print("Model training completed.")

# --- Free memory by deleting large training matrix ---
del X_train_final
gc.collect()

# --- Predict on test data ---
print("\nGenerating predictions for test set...")
preds_log = lgb_model.predict(X_test_final)
preds = np.expm1(preds_log)  # Convert back from log scale
preds[preds < 0] = 0        # Ensure no negative prices

# --- Prepare submission file ---
submission = pd.DataFrame({'sample_id': df_test['sample_id'], 'price': preds})
submission.to_csv('test_out.csv', index=False)

print("\nSubmission file 'test_out.csv' has been created successfully!")
print("Preview of first 5 predicted entries:")
print(submission.head())


Loading final training and testing data...
Data loaded successfully!
X_train shape: (75000, 1032)
X_test shape: (75000, 1032)

Training LightGBM model...
Model training complete.

Generating predictions on the test set...

Submission file 'test_out.csv' created successfully!
Here are the first 5 predictions:
   sample_id      price
0     100179  15.198210
1     245611  15.296555
2     146263  22.790701
3      95658  10.599428
4      36806  19.655482


In [3]:
# Cell 12: Hyperparameter Tuning with Optuna (CatBoost)

import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd
import pickle
import json
from datetime import datetime
from pathlib import Path
import os

# Create checkpoint directory
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Define SAVE_DIR if not already defined
if 'SAVE_DIR' not in locals():
    SAVE_DIR = Path("embeddings")

# Load data if not already in memory
if 'X_train' not in locals() or 'y_train' not in locals():
    print("Loading data...")
    X_train = np.load(SAVE_DIR / "final_X_train.npy", allow_pickle=True).astype(np.float32)
    X_test = np.load(SAVE_DIR / "final_X_test.npy", allow_pickle=True).astype(np.float32)

    if 'df_train' not in locals():
        # Assuming 'train.csv' is in the current directory
        df_train = pd.read_csv("train.csv")

    y_train = np.log1p(df_train['price'])

print(f"Data loaded: X_train shape = {X_train.shape}, y_train shape = {y_train.shape}")


def objective(trial):
    """Define the search space and model evaluation for Optuna with CatBoost."""
    params = {
        'iterations': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1e-9, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 20),
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'verbose': False,
        'thread_count': -1,
        'early_stopping_rounds': 50,
    }

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    rmse_scores = []

    for fold_idx, (train_index, val_index) in enumerate(kf.split(X_train)):
        X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
        y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

        model = CatBoostRegressor(**params)
        model.fit(
            X_train_fold,
            y_train_fold,
            eval_set=(X_val_fold, y_val_fold),
            use_best_model=True,
            verbose=False
        )

        preds = model.predict(X_val_fold)
        rmse = np.sqrt(mean_squared_error(y_val_fold, preds))
        rmse_scores.append(rmse)

        print(f"Trial {trial.number} - Fold {fold_idx+1}/3 - RMSE: {rmse:.6f}", end='\r')

    mean_rmse = np.mean(rmse_scores)
    print(f"Trial {trial.number} completed - Mean RMSE: {mean_rmse:.6f}" + " " * 20)
    return mean_rmse


# --- Run the optimization study ---
print("\nStarting hyperparameter optimization...")
print("Running 20 trials with 3-fold CV")
print("=" * 60)

study_name = f"catboost_optuna_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
storage_name = f"sqlite:///{CHECKPOINT_DIR}/optuna_study.db"

study = optuna.create_study(
    study_name=study_name,
    direction='minimize',
    storage=storage_name,
    load_if_exists=True
)


def checkpoint_callback(study, trial):
    checkpoint_file = f"{CHECKPOINT_DIR}/study_checkpoint.pkl"
    with open(checkpoint_file, 'wb') as f:
        pickle.dump(study, f)

    best_params_file = f"{CHECKPOINT_DIR}/best_params.json"
    results = {
        'best_trial_number': study.best_trial.number,
        'best_value': study.best_trial.value,
        'best_params': study.best_trial.params,
        'n_trials_completed': len(study.trials),
        'timestamp': datetime.now().isoformat()
    }
    with open(best_params_file, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"  → Checkpoint saved. Best RMSE so far: {study.best_trial.value:.6f}")


study.optimize(objective, n_trials=20, callbacks=[checkpoint_callback], show_progress_bar=True)

print("\n" + "=" * 60)
print("--- Best Trial Results ---")
print("=" * 60)
best_params = study.best_trial.params
print(f"Best Score (RMSE): {study.best_trial.value:.6f}")
print(f"Best Trial Number: {study.best_trial.number}")
print("\nBest Parameters:")
for key, value in best_params.items():
    if isinstance(value, float):
        print(f"  '{key}': {value:.6f},")
    else:
        print(f"  '{key}': {value},")

# --- Train final model with best parameters ---
print("\n" + "=" * 60)
print("Training Final Model with Best Parameters...")
print("=" * 60)

final_params = {
    'iterations': 2000,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'verbose': 100,
    'thread_count': -1,
    'early_stopping_rounds': 100,
}
# Update with best params, but keep the ones above fixed
final_params.update(best_params)


final_model = CatBoostRegressor(**final_params)
final_model.fit(
    X_train,
    y_train,
    verbose=100
)

print(f"\nFinal model trained with {final_model.tree_count_} trees")

# --- Make predictions ---
print("\nMaking predictions on test set...")

if 'df_test' not in locals():
    print("Loading test data...")
    df_test = pd.read_csv("test.csv")
    print(f"Test data loaded: {df_test.shape}")

y_pred_log = final_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

# --- Save predictions ---
print("\nSaving predictions...")

# Using 'sample_id' from the sample file for submission
if 'sample_id' in df_test.columns:
    test_ids = df_test['sample_id'].values
else:
    # Fallback to other possible ID columns or generate new ones
    id_columns = [col for col in df_test.columns if col.lower() in ['id', 'sample_id', 'test_id', 'index']]
    if id_columns:
        print(f"Using '{id_columns[0]}' as ID column.")
        test_ids = df_test[id_columns[0]].values
    else:
        print("Warning: 'sample_id' or other id columns not found. Creating sequential IDs.")
        test_ids = np.arange(len(y_pred))


submission = pd.DataFrame({
    'sample_id': test_ids,
    'price': y_pred
})

submission.to_csv('submission.csv', index=False)
print("Predictions saved to 'submission.csv'")
print(f"Submission shape: {submission.shape}")
print(f"Sample predictions:\n{submission.head()}")

[I 2025-10-11 23:14:45,012] A new study created in RDB with name: catboost_optuna_20251011_231444


Data loaded: X_train shape = (75000, 1032), y_train shape = (75000,)

Starting hyperparameter optimization...
Running 20 trials with 3-fold CV


  0%|          | 0/20 [00:00<?, ?it/s]

Trial 0 completed - Mean RMSE: 0.803230                    
[I 2025-10-11 23:16:45,051] Trial 0 finished with value: 0.8032304327298019 and parameters: {'learning_rate': 0.012736441930339923, 'depth': 8, 'l2_leaf_reg': 9.17479848494515, 'bagging_temperature': 0.12462495353147762, 'border_count': 104, 'random_strength': 3.8992585349928515, 'subsample': 0.8178700027642469, 'min_data_in_leaf': 5}. Best is trial 0 with value: 0.8032304327298019.
  → Checkpoint saved. Best RMSE so far: 0.803230
Trial 1 completed - Mean RMSE: 0.752616                    
[I 2025-10-11 23:18:22,291] Trial 1 finished with value: 0.7526157479311376 and parameters: {'learning_rate': 0.03352531203072505, 'depth': 7, 'l2_leaf_reg': 3.8686457377320345, 'bagging_temperature': 0.3487198021760879, 'border_count': 145, 'random_strength': 0.000383563303798599, 'subsample': 0.9723885642187399, 'min_data_in_leaf': 20}. Best is trial 1 with value: 0.7526157479311376.
  → Checkpoint saved. Best RMSE so far: 0.752616
Trial 2

In [5]:

# ULTIMATE ENSEMBLE - Maximum Performance Price Prediction
# This combines multiple advanced models with optimal hyperparameters

import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
import warnings
import gc
warnings.filterwarnings('ignore')

SAVE_DIR = Path("embeddings")

print("="*70)
print("🚀 ULTIMATE ENSEMBLE MODEL - MAXIMUM PERFORMANCE")
print("="*70)

# ============================================================================
# LOAD AND PREPARE DATA
# ============================================================================
print("\n[1/5] Loading data...")
X_train = np.load(SAVE_DIR / "final_X_train.npy", allow_pickle=True)
X_test = np.load(SAVE_DIR / "final_X_test.npy", allow_pickle=True)
y_train = df_train['price'].values

print(f"✓ X_train: {X_train.shape}")
print(f"✓ X_test: {X_test.shape}")
print(f"✓ y_train: {y_train.shape}")

# Clean data
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

# Log transform target
y_train_log = np.log1p(y_train)

print(f"\nPrice range: ${y_train.min():.2f} - ${y_train.max():.2f}")
print(f"Price mean: ${y_train.mean():.2f}, median: ${np.median(y_train):.2f}")

# ============================================================================
# K-FOLD CROSS VALIDATION SETUP
# ============================================================================
print(f"\n[2/5] Setting up 5-Fold Cross Validation...")
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Initialize prediction arrays
oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))
oof_et = np.zeros(len(X_train))

test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_et = np.zeros(len(X_test))

fold_scores = []

# ============================================================================
# TRAIN MODELS WITH K-FOLD
# ============================================================================
print(f"\n[3/5] Training ensemble models...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    print(f"\n{'─'*70}")
    print(f"📊 FOLD {fold}/{n_folds}")
    print(f"{'─'*70}")
    
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train_log[train_idx], y_train_log[val_idx]
    
    # ========================================================================
    # MODEL 1: LightGBM (Fast & Accurate)
    # ========================================================================
    print("Training LightGBM...")
    lgbm = lgb.LGBMRegressor(
        objective='regression',
        metric='mae',
        n_estimators=3000,
        learning_rate=0.01,
        num_leaves=31,
        max_depth=8,
        feature_fraction=0.75,
        bagging_fraction=0.75,
        bagging_freq=5,
        min_child_samples=20,
        min_child_weight=0.001,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1
    )
    lgbm.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    oof_lgb[val_idx] = lgbm.predict(X_val)
    test_lgb += lgbm.predict(X_test) / n_folds
    print(f"  ✓ Best iteration: {lgbm.best_iteration_}")
    
    # ========================================================================
    # MODEL 2: XGBoost (Robust)
    # ========================================================================
    print("Training XGBoost...")
    xgb = XGBRegressor(
        objective='reg:squarederror',
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=7,
        subsample=0.75,
        colsample_bytree=0.75,
        min_child_weight=1,
        reg_alpha=0.1,
        reg_lambda=0.1,
        gamma=0.01,
        random_state=42 + fold,
        n_jobs=-1,
        tree_method='hist',
        enable_categorical=False
    )
    xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = xgb.predict(X_val)
    test_xgb += xgb.predict(X_test) / n_folds
    print(f"  ✓ Completed")
    
    # ========================================================================
    # MODEL 3: CatBoost (Powerful)
    # ========================================================================
    print("Training CatBoost...")
    cat = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.01,
        depth=7,
        l2_leaf_reg=3,
        min_data_in_leaf=20,
        random_strength=0.5,
        bagging_temperature=0.2,
        od_type='Iter',
        od_wait=100,
        random_seed=42 + fold,
        verbose=False,
        task_type='CPU',
        thread_count=-1
    )
    cat.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )
    oof_cat[val_idx] = cat.predict(X_val)
    test_cat += cat.predict(X_test) / n_folds
    print(f"  ✓ Best iteration: {cat.best_iteration_}")
    
    # ========================================================================
    # MODEL 4: Extra Trees (Diversity)
    # ========================================================================
    print("Training Extra Trees...")
    et = ExtraTreesRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42 + fold,
        n_jobs=-1,
        verbose=0
    )
    et.fit(X_tr, y_tr)
    oof_et[val_idx] = et.predict(X_val)
    test_et += et.predict(X_test) / n_folds
    print(f"  ✓ Completed")
    
    # ========================================================================
    # Calculate Fold Performance
    # ========================================================================
    fold_pred = (oof_lgb[val_idx] + oof_xgb[val_idx] + 
                 oof_cat[val_idx] + oof_et[val_idx]) / 4
    fold_actual = np.expm1(y_val)
    fold_pred_price = np.expm1(fold_pred)
    
    # SMAPE
    fold_smape = np.mean(
        np.abs(fold_pred_price - fold_actual) / 
        ((np.abs(fold_actual) + np.abs(fold_pred_price)) / 2)
    ) * 100
    
    fold_scores.append(fold_smape)
    print(f"\n  Fold {fold} SMAPE: {fold_smape:.4f}%")
    
    # Memory cleanup
    del X_tr, X_val, y_tr, y_val, lgbm, xgb, cat, et
    gc.collect()

# ============================================================================
# STACKING LAYER (META MODEL)
# ============================================================================
print(f"\n[4/5] Training stacking meta-model...")

# Prepare stacking features
stack_train = np.column_stack([oof_lgb, oof_xgb, oof_cat, oof_et])
stack_test = np.column_stack([test_lgb, test_xgb, test_cat, test_et])

# Meta model (Ridge regression for stability)
meta_model = Ridge(alpha=10.0, random_state=42)
meta_model.fit(stack_train, y_train_log)

# Get meta predictions
meta_train_pred = meta_model.predict(stack_train)
meta_test_pred = meta_model.predict(stack_test)

print(f"✓ Stacking weights: {meta_model.coef_.round(3)}")

# ============================================================================
# FINAL ENSEMBLE (Weighted Average + Stacking)
# ============================================================================
print(f"\n[5/5] Creating final ensemble predictions...")

# Optimized weights based on typical performance
# LGB: 0.30, XGB: 0.28, CAT: 0.27, ET: 0.15
final_avg = (0.30 * test_lgb + 0.28 * test_xgb + 
             0.27 * test_cat + 0.15 * test_et)

# Blend weighted average with meta model (70-30 split)
final_pred_log = 0.70 * final_avg + 0.30 * meta_test_pred

# Convert back to original price scale
predictions = np.expm1(final_pred_log)
predictions = np.clip(predictions, 0.01, None)  # Ensure positive prices

# ============================================================================
# EVALUATION & RESULTS
# ============================================================================
print("\n" + "="*70)
print("CROSS-VALIDATION RESULTS")
print("="*70)

for i, score in enumerate(fold_scores, 1):
    print(f"Fold {i}: {score:.4f}%")

overall_smape = np.mean(
    np.abs(np.expm1(meta_train_pred) - np.expm1(y_train_log)) / 
    ((np.abs(np.expm1(y_train_log)) + np.abs(np.expm1(meta_train_pred))) / 2)
) * 100

print(f"\n{'─'*70}")
print(f" Mean CV SMAPE: {np.mean(fold_scores):.4f}% (±{np.std(fold_scores):.4f})")
print(f" Overall SMAPE: {overall_smape:.4f}%")
print(f"{'─'*70}")

# ============================================================================
# CREATE SUBMISSION FILE
# ============================================================================
submission = pd.DataFrame({
    'sample_id': df_test['sample_id'].values,
    'price': predictions
})

submission.to_csv('test_out.csv', index=False)

print("\n" + "="*70)
print(" SUBMISSION FILE CREATED: test_out.csv")
print("="*70)

print("\nPrediction Statistics:")
print(submission['price'].describe())

print("\nSample Predictions:")
print(submission.head(10))

print("\n DONE! Your submission is ready for upload!")
print("="*70)

🚀 ULTIMATE ENSEMBLE MODEL - MAXIMUM PERFORMANCE

[1/5] Loading data...
✓ X_train: (75000, 1032)
✓ X_test: (75000, 1032)
✓ y_train: (75000,)

Price range: $0.13 - $2796.00
Price mean: $23.65, median: $14.00

[2/5] Setting up 5-Fold Cross Validation...

[3/5] Training ensemble models...

──────────────────────────────────────────────────────────────────────
📊 FOLD 1/5
──────────────────────────────────────────────────────────────────────
Training LightGBM...
  ✓ Best iteration: 2996
Training XGBoost...
  ✓ Completed
Training CatBoost...
  ✓ Best iteration: 2999
Training Extra Trees...
  ✓ Completed

  Fold 1 SMAPE: 60.1387%

──────────────────────────────────────────────────────────────────────
📊 FOLD 2/5
──────────────────────────────────────────────────────────────────────
Training LightGBM...
  ✓ Best iteration: 3000
Training XGBoost...
  ✓ Completed
Training CatBoost...
  ✓ Best iteration: 2999
Training Extra Trees...
  ✓ Completed

  Fold 2 SMAPE: 59.3842%

─────────────────────────

In [8]:
# Simple but Highly Optimized LightGBM Solution
# Clean code with proven feature engineering techniques

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# FEATURE EXTRACTION
# ============================================================================

# Global variable to store encodings
TARGET_ENCODINGS = {}

def extract_features(df, embeddings, prices=None, is_train=True):
    """Extract comprehensive features from data"""
    
    global TARGET_ENCODINGS
    
    print(f"Extracting features for {len(df)} samples...")
    
    # Initialize features dict
    features = {}
    
    # Text content
    text = df['catalog_content'].str.lower()
    
    # =========================
    # BASIC TEXT FEATURES
    # =========================
    features['text_length'] = text.str.len()
    features['word_count'] = text.str.split().str.len()
    features['char_count'] = text.str.len()
    features['digit_count'] = text.str.count(r'\d')
    features['digit_ratio'] = features['digit_count'] / (features['text_length'] + 1)
    
    # =========================
    # IPQ (ITEM PACK QUANTITY)
    # =========================
    # This is crucial for pricing!
    ipq = text.str.extract(r'(?:pack|ipq)[:\s]*(\d+)', expand=False)
    features['ipq'] = ipq.fillna(1).astype(float)
    features['log_ipq'] = np.log1p(features['ipq'])
    features['has_pack'] = (features['ipq'] > 1).astype(int)
    
    # =========================
    # SIZE/QUANTITY EXTRACTION
    # =========================
    features['ml_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*ml', expand=False).fillna(0).astype(float)
    features['gram_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:g|gram)', expand=False).fillna(0).astype(float)
    features['kg_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*kg', expand=False).fillna(0).astype(float)
    features['liter_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:l|liter)', expand=False).fillna(0).astype(float)
    
    # Total quantity (normalized)
    features['total_qty'] = (
        features['ml_amount'] + 
        features['gram_amount'] * 1000 + 
        features['kg_amount'] * 1000000 +
        features['liter_amount'] * 1000
    )
    features['log_total_qty'] = np.log1p(features['total_qty'])
    
    # =========================
    # BRAND INDICATORS
    # =========================
    premium_brands = ['apple', 'samsung', 'sony', 'lg', 'nike', 'adidas', 'canon', 'nikon']
    features['is_premium_brand'] = text.apply(lambda x: any(b in x for b in premium_brands)).astype(int)
    
    # =========================
    # CATEGORY INDICATORS
    # =========================
    features['is_electronics'] = text.str.contains('electronic|digital|wireless|bluetooth|usb', regex=True).astype(int)
    features['is_clothing'] = text.str.contains('shirt|pant|dress|jacket|cotton', regex=True).astype(int)
    features['is_food'] = text.str.contains('food|snack|nutrition|organic', regex=True).astype(int)
    features['is_beauty'] = text.str.contains('beauty|cosmetic|skincare|cream', regex=True).astype(int)
    features['is_sports'] = text.str.contains('sport|fitness|gym|yoga', regex=True).astype(int)
    
    # =========================
    # QUALITY INDICATORS
    # =========================
    quality_words = ['premium', 'deluxe', 'luxury', 'pro', 'ultra', 'advanced']
    features['quality_score'] = text.apply(lambda x: sum(1 for w in quality_words if w in x))
    features['is_premium_quality'] = (features['quality_score'] > 0).astype(int)
    
    # =========================
    # NUMERIC FEATURES
    # =========================
    def extract_max_number(txt):
        numbers = re.findall(r'\d+', str(txt))
        return max([int(n) for n in numbers]) if numbers else 0
    
    features['max_number'] = text.apply(extract_max_number)
    features['log_max_number'] = np.log1p(features['max_number'])
    features['num_count'] = text.str.findall(r'\d+').apply(len)
    
    # =========================
    # EMBEDDING STATISTICS
    # =========================
    features['emb_mean'] = embeddings.mean(axis=1)
    features['emb_std'] = embeddings.std(axis=1)
    features['emb_max'] = embeddings.max(axis=1)
    features['emb_min'] = embeddings.min(axis=1)
    features['emb_range'] = features['emb_max'] - features['emb_min']
    features['emb_median'] = np.median(embeddings, axis=1)
    
    # =========================
    # INTERACTION FEATURES
    # =========================
    features['ipq_x_total_qty'] = features['ipq'] * features['total_qty']
    features['brand_x_quality'] = features['is_premium_brand'] * features['quality_score']
    features['category_count'] = (
        features['is_electronics'] + features['is_clothing'] + 
        features['is_food'] + features['is_beauty'] + features['is_sports']
    )
    
    # Convert to DataFrame
    features_df = pd.DataFrame(features)
    
    # =========================
    # TARGET ENCODING
    # =========================
    if is_train and prices is not None:
        # Create encodings and store them
        global_mean = np.log1p(prices).mean()
        TARGET_ENCODINGS['global_mean'] = global_mean
        
        for col in ['is_electronics', 'is_clothing', 'is_food', 'is_beauty']:
            try:
                temp_df = pd.DataFrame({'cat': features_df[col], 'price': prices})
                encoding = temp_df.groupby('cat')['price'].mean().to_dict()
                TARGET_ENCODINGS[col] = encoding
                features_df[f'{col}_target_enc'] = features_df[col].map(encoding).fillna(global_mean)
            except:
                TARGET_ENCODINGS[col] = {}
                features_df[f'{col}_target_enc'] = global_mean
    else:
        # Use stored encodings for test data
        global_mean = TARGET_ENCODINGS.get('global_mean', 0)
        for col in ['is_electronics', 'is_clothing', 'is_food', 'is_beauty']:
            encoding = TARGET_ENCODINGS.get(col, {})
            features_df[f'{col}_target_enc'] = features_df[col].map(encoding).fillna(global_mean)
    
    # =========================
    # COMBINE WITH EMBEDDINGS
    # =========================
    X_combined = np.hstack([embeddings, features_df.values])
    
    print(f"Total features: {X_combined.shape[1]}")
    return X_combined

# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def train_model():
    """Main training pipeline"""
    
    # Load data
    print("=" * 80)
    print("LOADING DATA")
    print("=" * 80)
    
    SAVE_DIR = Path("embeddings")
    X_train_emb = np.load(SAVE_DIR / "final_X_train.npy", allow_pickle=True).astype(np.float32)
    X_test_emb = np.load(SAVE_DIR / "final_X_test.npy", allow_pickle=True).astype(np.float32)
    
    df_train = pd.read_csv("train.csv")
    df_test = pd.read_csv("test.csv")
    
    print(f"Train shape: {df_train.shape}")
    print(f"Test shape: {df_test.shape}")
    
    # Extract features
    print("\n" + "=" * 80)
    print("FEATURE ENGINEERING")
    print("=" * 80)
    
    X_train = extract_features(df_train, X_train_emb, df_train['price'].values, is_train=True)
    X_test = extract_features(df_test, X_test_emb, is_train=False)
    
    # Target variable (log-transformed)
    y_train = np.log1p(df_train['price'].values)
    
    # Optimized LightGBM parameters
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.01,
        'num_leaves': 255,
        'max_depth': 12,
        'min_child_samples': 20,
        'subsample': 0.85,
        'subsample_freq': 1,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'min_split_gain': 0.001,
        'verbose': -1,
        'random_state': 42,
        'n_jobs': -1,
    }
    
    # Cross-validation
    print("\n" + "=" * 80)
    print("CROSS-VALIDATION TRAINING")
    print("=" * 80)
    
    n_splits = 5
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    oof_predictions = np.zeros(len(X_train))
    test_predictions = np.zeros(len(X_test))
    fold_scores = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"\nFold {fold_idx + 1}/{n_splits}")
        print("-" * 80)
        
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # Create datasets
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        # Train
        model = lgb.train(
            params,
            train_data,
            num_boost_round=5000,
            valid_sets=[val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=200)
            ]
        )
        
        # Predict
        oof_predictions[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
        test_predictions += model.predict(X_test, num_iteration=model.best_iteration) / n_splits
        
        # Calculate SMAPE
        y_val_actual = np.expm1(y_val)
        y_pred_actual = np.expm1(oof_predictions[val_idx])
        
        denominator = (np.abs(y_val_actual) + np.abs(y_pred_actual)) / 2.0
        diff = np.abs(y_val_actual - y_pred_actual)
        smape = np.mean(np.where(denominator == 0, 0, diff / denominator)) * 100
        
        fold_scores.append(smape)
        print(f"Fold {fold_idx + 1} SMAPE: {smape:.4f}%")
    
    # Final results
    print("\n" + "=" * 80)
    print("FINAL RESULTS")
    print("=" * 80)
    print(f"Mean CV SMAPE: {np.mean(fold_scores):.4f}% (+/- {np.std(fold_scores):.4f}%)")
    print(f"Fold scores: {[f'{s:.4f}%' for s in fold_scores]}")
    
    # Convert predictions back to original scale
    y_pred_final = np.expm1(test_predictions)
    
    # Ensure positive prices
    y_pred_final = np.maximum(y_pred_final, 0.01)
    
    # Create submission
    submission = pd.DataFrame({
        'sample_id': df_test['sample_id'].values,
        'price': y_pred_final
    })
    
    submission.to_csv('submission_simple_optimized.csv', index=False)
    
    print("\n" + "=" * 80)
    print("SUBMISSION CREATED")
    print("=" * 80)
    print(f"✓ File: submission_simple_optimized.csv")
    print(f"✓ Shape: {submission.shape}")
    print(f"\nPrice Statistics:")
    print(f"  Min:    ${submission['price'].min():.2f}")
    print(f"  Max:    ${submission['price'].max():.2f}")
    print(f"  Mean:   ${submission['price'].mean():.2f}")
    print(f"  Median: ${submission['price'].median():.2f}")
    print(f"  Std:    ${submission['price'].std():.2f}")
    
    return submission

# ============================================================================
# RUN
# ============================================================================

if __name__ == "__main__":
    submission = train_model()

LOADING DATA
Train shape: (75000, 4)
Test shape: (75000, 3)

FEATURE ENGINEERING
Extracting features for 75000 samples...
Total features: 1070
Extracting features for 75000 samples...
Total features: 1070

CROSS-VALIDATION TRAINING

Fold 1/5
--------------------------------------------------------------------------------
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.782278
[400]	valid_0's rmse: 0.752712
[600]	valid_0's rmse: 0.741191
[800]	valid_0's rmse: 0.735134
[1000]	valid_0's rmse: 0.731304
[1200]	valid_0's rmse: 0.728919
[1400]	valid_0's rmse: 0.727431
[1600]	valid_0's rmse: 0.726253
[1800]	valid_0's rmse: 0.725337
[2000]	valid_0's rmse: 0.72455
[2200]	valid_0's rmse: 0.723917
[2400]	valid_0's rmse: 0.723389
[2600]	valid_0's rmse: 0.722966
[2800]	valid_0's rmse: 0.722685
[3000]	valid_0's rmse: 0.722449
[3200]	valid_0's rmse: 0.722231
[3400]	valid_0's rmse: 0.722044
[3600]	valid_0's rmse: 0.721864
[3800]	valid_0's rmse: 0.721745
[4000]	valid

In [1]:
"""
Ultra-Optimized Ensemble Price Predictor for M1 MacBook Pro
Uses CatBoost + XGBoost + LightGBM + Neural Network Stacking
Advanced feature engineering with pseudo-labeling and TTA
"""

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# M1 optimization
import os
os.environ['OMP_NUM_THREADS'] = '8'
os.environ['MKL_NUM_THREADS'] = '8'

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    SEED = 42
    N_SPLITS = 5  # Reduced from 7
    N_THREADS = 8
    
    # Model parameters - SPEED OPTIMIZED
    LGB_PARAMS = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.03,  # Increased from 0.008
        'num_leaves': 200,  # Reduced from 300
        'max_depth': 10,  # Reduced from 14
        'min_child_samples': 20,
        'subsample': 0.85,
        'subsample_freq': 1,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'min_split_gain': 0.001,
        'verbose': -1,
        'random_state': 42,
        'n_jobs': 8,
    }
    
    XGB_PARAMS = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': 0.03,  # Increased from 0.008
        'max_depth': 8,  # Reduced from 12
        'min_child_weight': 3,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'gamma': 0.01,
        'tree_method': 'auto',
        'random_state': 42,
        'n_jobs': 8,
    }
    
    CAT_PARAMS = {
        'loss_function': 'RMSE',
        'learning_rate': 0.03,  # Increased from 0.008
        'depth': 8,  # Reduced from 11
        'l2_leaf_reg': 3.0,
        'bootstrap_type': 'Bayesian',
        'bagging_temperature': 0.5,
        'random_strength': 0.5,
        'iterations': 2000,  # Reduced from 5000
        'early_stopping_rounds': 100,  # Reduced from 150
        'random_seed': 42,
        'thread_count': 8,
        'verbose': False,
        'task_type': 'CPU',
    }

# ============================================================================
# GLOBAL STORAGE
# ============================================================================

TARGET_ENCODINGS = {}
SCALERS = {}
CLUSTERS = {}
SVD_MODELS = {}

# ============================================================================
# ADVANCED FEATURE EXTRACTION
# ============================================================================

def extract_ultra_features(df, embeddings, prices=None, is_train=True):
    """Ultra-comprehensive feature extraction"""
    
    global TARGET_ENCODINGS, SCALERS, CLUSTERS, SVD_MODELS
    
    print(f"Extracting features for {len(df)} samples...")
    
    # Clean embeddings first - replace any NaN/inf values
    embeddings = np.nan_to_num(embeddings, nan=0.0, posinf=0.0, neginf=0.0)
    
    features = {}
    text = df['catalog_content'].str.lower()
    text_original = df['catalog_content']
    
    # =========================
    # TEXT STATISTICS (Enhanced)
    # =========================
    features['text_length'] = text.str.len()
    features['word_count'] = text.str.split().str.len()
    features['avg_word_length'] = features['text_length'] / (features['word_count'] + 1)
    features['digit_count'] = text.str.count(r'\d')
    features['digit_ratio'] = features['digit_count'] / (features['text_length'] + 1)
    features['upper_count'] = text_original.str.count(r'[A-Z]')
    features['upper_ratio'] = features['upper_count'] / (features['text_length'] + 1)
    features['special_char_count'] = text.str.count(r'[^\w\s]')
    features['comma_count'] = text.str.count(',')
    features['space_count'] = text.str.count(' ')
    features['dot_count'] = text.str.count(r'\.')
    features['dash_count'] = text.str.count('-')
    features['bracket_count'] = text.str.count(r'[\(\)\[\]]')
    features['quote_count'] = text.str.count(r'["\']')
    
    # Sentence structure
    features['sentence_count'] = text.str.count(r'[.!?]') + 1
    features['words_per_sentence'] = features['word_count'] / features['sentence_count']
    features['unique_word_ratio'] = text.str.split().apply(lambda x: len(set(x)) / (len(x) + 1))
    
    # =========================
    # IPQ EXTRACTION (Multi-Pattern)
    # =========================
    ipq_patterns = [
        r'(?:pack|ipq)[:\s]*(\d+)',
        r'(\d+)\s*(?:pack|pcs|pieces|piece)',
        r'pack\s*of\s*(\d+)',
        r'set\s*of\s*(\d+)',
        r'(\d+)\s*count',
        r'quantity[:\s]*(\d+)',
    ]
    
    ipq_extracts = []
    for pattern in ipq_patterns:
        extracted = text.str.extract(pattern, expand=False)
        ipq_extracts.append(extracted)
    
    ipq_df = pd.concat(ipq_extracts, axis=1)
    features['ipq'] = ipq_df.bfill(axis=1).iloc[:, 0].fillna(1).astype(float)
    features['ipq'] = np.clip(features['ipq'], 1, 1000)  # Clip outliers
    
    features['log_ipq'] = np.log1p(features['ipq'])
    features['sqrt_ipq'] = np.sqrt(features['ipq'])
    features['cbrt_ipq'] = np.cbrt(features['ipq'])
    features['ipq_squared'] = features['ipq'] ** 2
    features['ipq_cubed'] = features['ipq'] ** 3
    features['ipq_inv'] = 1 / (features['ipq'] + 0.01)
    
    # IPQ bins
    features['ipq_bin_1'] = (features['ipq'] == 1).astype(int)
    features['ipq_bin_2_3'] = ((features['ipq'] >= 2) & (features['ipq'] <= 3)).astype(int)
    features['ipq_bin_4_6'] = ((features['ipq'] >= 4) & (features['ipq'] <= 6)).astype(int)
    features['ipq_bin_7_12'] = ((features['ipq'] >= 7) & (features['ipq'] <= 12)).astype(int)
    features['ipq_bin_13plus'] = (features['ipq'] >= 13).astype(int)
    
    # =========================
    # QUANTITY EXTRACTION (All Units)
    # =========================
    # Volume
    features['ml_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*ml\b', expand=False).fillna(0).astype(float)
    features['liter_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:l|liter|litre)\b', expand=False).fillna(0).astype(float)
    features['oz_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:oz|ounce)\b', expand=False).fillna(0).astype(float)
    features['gallon_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:gal|gallon)\b', expand=False).fillna(0).astype(float)
    
    # Weight
    features['gram_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:g|gm|gram)\b', expand=False).fillna(0).astype(float)
    features['kg_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:kg|kilogram)\b', expand=False).fillna(0).astype(float)
    features['lb_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:lb|lbs|pound)\b', expand=False).fillna(0).astype(float)
    features['mg_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*mg\b', expand=False).fillna(0).astype(float)
    
    # Length
    features['cm_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*cm\b', expand=False).fillna(0).astype(float)
    features['mm_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*mm\b', expand=False).fillna(0).astype(float)
    features['inch_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:inch|in)\b', expand=False).fillna(0).astype(float)
    features['meter_amount'] = text.str.extract(r'(\d+(?:\.\d+)?)\s*(?:m|meter)\b', expand=False).fillna(0).astype(float)
    
    # Normalize to standard units
    features['total_volume'] = (
        features['ml_amount'] + 
        features['liter_amount'] * 1000 +
        features['oz_amount'] * 29.5735 +
        features['gallon_amount'] * 3785.41
    )
    
    features['total_weight'] = (
        features['gram_amount'] + 
        features['kg_amount'] * 1000 +
        features['lb_amount'] * 453.592 +
        features['mg_amount'] * 0.001
    )
    
    features['total_length'] = (
        features['cm_amount'] +
        features['mm_amount'] * 0.1 +
        features['inch_amount'] * 2.54 +
        features['meter_amount'] * 100
    )
    
    # Unified quantity
    features['total_qty'] = features['total_volume'] + features['total_weight'] + features['total_length']
    features['log_total_qty'] = np.log1p(features['total_qty'])
    features['sqrt_total_qty'] = np.sqrt(features['total_qty'])
    features['cbrt_total_qty'] = np.cbrt(features['total_qty'])
    
    # Per pack metrics
    features['qty_per_pack'] = features['total_qty'] / (features['ipq'] + 0.01)
    features['log_qty_per_pack'] = np.log1p(features['qty_per_pack'])
    features['volume_per_pack'] = features['total_volume'] / (features['ipq'] + 0.01)
    features['weight_per_pack'] = features['total_weight'] / (features['ipq'] + 0.01)
    
    # Quantity indicators
    features['has_volume'] = (features['total_volume'] > 0).astype(int)
    features['has_weight'] = (features['total_weight'] > 0).astype(int)
    features['has_length'] = (features['total_length'] > 0).astype(int)
    features['has_any_qty'] = (features['total_qty'] > 0).astype(int)
    features['qty_type_count'] = features['has_volume'] + features['has_weight'] + features['has_length']
    
    # =========================
    # BRAND DETECTION (Comprehensive)
    # =========================
    brand_dict = {
        'tech_premium': ['apple', 'samsung', 'sony', 'lg', 'dell', 'hp', 'lenovo', 'asus', 'microsoft', 'intel', 'amd'],
        'tech_mid': ['xiaomi', 'oneplus', 'oppo', 'vivo', 'realme', 'honor', 'motorola'],
        'appliance': ['bosch', 'philips', 'braun', 'dyson', 'panasonic', 'whirlpool', 'electrolux'],
        'fashion_premium': ['nike', 'adidas', 'puma', 'reebok', 'under armour', 'levi', 'calvin klein'],
        'fashion_fast': ['h&m', 'zara', 'uniqlo', 'gap', 'forever 21'],
        'beauty_premium': ['loreal', 'maybelline', 'revlon', 'clinique', 'estee lauder', 'mac', 'sephora'],
        'beauty_mass': ['nivea', 'garnier', 'dove', 'vaseline', 'olay'],
    }
    
    for brand_cat, brands in brand_dict.items():
        features[f'is_{brand_cat}'] = text.apply(lambda x: any(b in x for b in brands)).astype(int)
    
    features['brand_tier'] = (
        features['is_tech_premium'] * 3 + features['is_fashion_premium'] * 3 + 
        features['is_beauty_premium'] * 3 + features['is_tech_mid'] * 2 +
        features['is_fashion_fast'] + features['is_beauty_mass']
    )
    
    # =========================
    # CATEGORY DETECTION (Extended)
    # =========================
    categories = {
        'electronics': r'electronic|digital|wireless|bluetooth|usb|hdmi|led|lcd|smart|wifi|tech|gadget|device',
        'computer': r'computer|laptop|desktop|tablet|monitor|keyboard|mouse|processor|ram|ssd',
        'phone': r'phone|mobile|smartphone|iphone|android|sim|cellular',
        'audio': r'headphone|earphone|speaker|audio|sound|music|mic|microphone',
        'camera': r'camera|lens|photo|video|tripod|flash|dslr|mirrorless',
        'clothing': r'shirt|pant|dress|jacket|cotton|polyester|fabric|wear|apparel|sleeve|collar',
        'shoes': r'shoe|sneaker|boot|sandal|slipper|footwear|sole',
        'food': r'food|snack|nutrition|organic|protein|vitamin|flavor|taste|edible|ingredient',
        'beverage': r'drink|beverage|juice|water|coffee|tea|soda|beer|wine',
        'beauty': r'beauty|cosmetic|skincare|cream|lotion|serum|moisturizer|cleanser|makeup|lipstick',
        'health': r'health|medical|vitamin|supplement|medicine|pharmaceutical|wellness|care',
        'sports': r'sport|fitness|gym|yoga|exercise|workout|training|athletic|dumbell',
        'home': r'home|kitchen|dining|furniture|decor|bedding|bath|storage|appliance',
        'toys': r'toy|game|play|kids|children|puzzle|doll|action figure|lego',
        'books': r'book|novel|paperback|hardcover|author|edition|publication|reading',
        'automotive': r'car|auto|vehicle|motor|engine|tire|brake|oil|battery',
        'tools': r'tool|drill|hammer|screwdriver|wrench|saw|equipment',
        'garden': r'garden|plant|seed|fertilizer|lawn|outdoor|greenhouse',
        'pet': r'pet|dog|cat|animal|bird|fish|aquarium',
        'office': r'office|stationery|pen|pencil|notebook|folder|desk|chair',
    }
    
    for cat_name, pattern in categories.items():
        features[f'is_{cat_name}'] = text.str.contains(pattern, regex=True).astype(int)
    
    features['category_count'] = sum(features[f'is_{cat}'] for cat in categories.keys())
    features['is_multi_category'] = (features['category_count'] > 1).astype(int)
    
    # Category groups
    features['is_tech_any'] = (features['is_electronics'] | features['is_computer'] | features['is_phone'] | features['is_audio']).astype(int)
    features['is_fashion_any'] = (features['is_clothing'] | features['is_shoes']).astype(int)
    features['is_consumable'] = (features['is_food'] | features['is_beverage'] | features['is_beauty'] | features['is_health']).astype(int)
    
    # =========================
    # QUALITY & PRICE SIGNALS
    # =========================
    premium_words = ['premium', 'deluxe', 'luxury', 'pro', 'ultra', 'advanced', 'professional', 
                     'high-end', 'superior', 'exclusive', 'elite', 'platinum', 'gold', 'ultimate',
                     'supreme', 'prestige', 'signature', 'limited edition', 'special edition']
    
    budget_words = ['cheap', 'budget', 'economy', 'basic', 'standard', 'value', 'affordable',
                    'generic', 'simple', 'essential', 'everyday']
    
    quality_words = ['quality', 'durable', 'premium', 'certified', 'authentic', 'original', 
                     'genuine', 'tested', 'approved', 'guaranteed', 'warranty']
    
    new_words = ['new', 'latest', 'newest', 'fresh', 'brand new', '2024', '2025']
    
    features['premium_word_count'] = text.apply(lambda x: sum(w in x for w in premium_words))
    features['budget_word_count'] = text.apply(lambda x: sum(w in x for w in budget_words))
    features['quality_word_count'] = text.apply(lambda x: sum(w in x for w in quality_words))
    features['new_word_count'] = text.apply(lambda x: sum(w in x for w in new_words))
    
    features['quality_score'] = (
        features['premium_word_count'] * 3 + 
        features['quality_word_count'] * 2 + 
        features['new_word_count'] - 
        features['budget_word_count'] * 2
    )
    
    features['has_premium'] = (features['premium_word_count'] > 0).astype(int)
    features['has_budget'] = (features['budget_word_count'] > 0).astype(int)
    features['has_warranty'] = text.str.contains('warranty|guarantee', regex=True).astype(int)
    
    # =========================
    # MATERIAL & CONSTRUCTION
    # =========================
    materials = {
        'metal': r'metal|steel|aluminum|aluminium|brass|copper|iron|titanium',
        'plastic': r'plastic|polymer|pvc|abs',
        'wood': r'wood|wooden|timber|oak|pine|bamboo',
        'glass': r'glass|crystal|tempered',
        'leather': r'leather|suede|genuine leather',
        'cotton': r'cotton|fabric|textile|cloth',
        'ceramic': r'ceramic|porcelain|clay',
        'rubber': r'rubber|silicone|latex',
        'carbon': r'carbon fiber|graphite',
    }
    
    for mat_name, pattern in materials.items():
        features[f'is_{mat_name}'] = text.str.contains(pattern, regex=True).astype(int)
    
    features['material_count'] = sum(features[f'is_{mat}'] for mat in materials.keys())
    features['is_premium_material'] = (features['is_metal'] | features['is_leather'] | features['is_glass'] | features['is_carbon']).astype(int)
    
    # =========================
    # COLOR DETECTION
    # =========================
    colors = ['black', 'white', 'red', 'blue', 'green', 'yellow', 'pink', 'purple', 'orange', 
              'brown', 'grey', 'gray', 'silver', 'gold', 'beige']
    
    features['color_count'] = text.apply(lambda x: sum(1 for c in colors if c in x))
    features['has_color'] = (features['color_count'] > 0).astype(int)
    features['has_multiple_colors'] = (features['color_count'] > 1).astype(int)
    
    # Specific colors
    features['is_black'] = text.str.contains(r'\bblack\b').astype(int)
    features['is_white'] = text.str.contains(r'\bwhite\b').astype(int)
    features['is_metallic'] = text.str.contains('silver|gold|metallic').astype(int)
    
    # =========================
    # NUMERIC PATTERNS (Advanced)
    # =========================
    def extract_all_numbers(txt):
        nums = re.findall(r'\d+(?:\.\d+)?', str(txt))
        return [float(n) for n in nums] if nums else [0]
    
    all_numbers = text.apply(extract_all_numbers)
    
    features['num_count'] = all_numbers.apply(len)
    features['max_number'] = all_numbers.apply(max)
    features['min_number'] = all_numbers.apply(min)
    features['mean_number'] = all_numbers.apply(np.mean)
    features['median_number'] = all_numbers.apply(np.median)
    features['std_number'] = all_numbers.apply(lambda x: np.std(x) if len(x) > 1 else 0)
    features['sum_numbers'] = all_numbers.apply(sum)
    features['range_numbers'] = features['max_number'] - features['min_number']
    
    # Log transforms
    features['log_max_number'] = np.log1p(features['max_number'])
    features['log_sum_numbers'] = np.log1p(features['sum_numbers'])
    features['log_mean_number'] = np.log1p(features['mean_number'])
    
    # Number bins
    features['has_large_number'] = (features['max_number'] > 100).astype(int)
    features['has_very_large_number'] = (features['max_number'] > 1000).astype(int)
    
    # =========================
    # EMBEDDING FEATURES (Comprehensive)
    # =========================
    # Basic statistics
    features['emb_mean'] = embeddings.mean(axis=1)
    features['emb_std'] = embeddings.std(axis=1)
    features['emb_max'] = embeddings.max(axis=1)
    features['emb_min'] = embeddings.min(axis=1)
    features['emb_range'] = features['emb_max'] - features['emb_min']
    features['emb_median'] = np.median(embeddings, axis=1)
    features['emb_q10'] = np.percentile(embeddings, 10, axis=1)
    features['emb_q25'] = np.percentile(embeddings, 25, axis=1)
    features['emb_q75'] = np.percentile(embeddings, 75, axis=1)
    features['emb_q90'] = np.percentile(embeddings, 90, axis=1)
    features['emb_iqr'] = features['emb_q75'] - features['emb_q25']
    features['emb_cv'] = features['emb_std'] / (np.abs(features['emb_mean']) + 1e-6)
    
    # Advanced statistics
    emb_df = pd.DataFrame(embeddings)
    features['emb_skew'] = emb_df.skew(axis=1).values
    features['emb_kurtosis'] = emb_df.kurtosis(axis=1).values
    
    # Positive/negative analysis
    features['emb_sum_positive'] = (embeddings * (embeddings > 0)).sum(axis=1)
    features['emb_sum_negative'] = (embeddings * (embeddings < 0)).sum(axis=1)
    features['emb_count_positive'] = (embeddings > 0).sum(axis=1)
    features['emb_count_negative'] = (embeddings < 0).sum(axis=1)
    features['emb_ratio_positive'] = features['emb_count_positive'] / embeddings.shape[1]
    features['emb_ratio_negative'] = features['emb_count_negative'] / embeddings.shape[1]
    
    # Magnitude features
    features['emb_l1_norm'] = np.abs(embeddings).sum(axis=1)
    features['emb_l2_norm'] = np.sqrt((embeddings ** 2).sum(axis=1))
    features['emb_mean_abs'] = np.abs(embeddings).mean(axis=1)
    
    # Key dimensions
    for i in [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75]:
        if i < embeddings.shape[1]:
            features[f'emb_dim_{i}'] = embeddings[:, i]
    
    # SVD/PCA of embeddings
    if is_train:
        # Check and clean embeddings for NaN/inf
        embeddings_clean = np.copy(embeddings)
        embeddings_clean = np.nan_to_num(embeddings_clean, nan=0.0, posinf=0.0, neginf=0.0)
        
        n_components = min(20, embeddings_clean.shape[1])
        svd = TruncatedSVD(n_components=n_components, random_state=42)
        emb_svd = svd.fit_transform(embeddings_clean)
        SVD_MODELS['emb_svd'] = svd
    else:
        svd = SVD_MODELS.get('emb_svd')
        if svd is not None:
            embeddings_clean = np.copy(embeddings)
            embeddings_clean = np.nan_to_num(embeddings_clean, nan=0.0, posinf=0.0, neginf=0.0)
            emb_svd = svd.transform(embeddings_clean)
        else:
            emb_svd = embeddings[:, :20]
    
    for i in range(emb_svd.shape[1]):
        features[f'emb_svd_{i}'] = emb_svd[:, i]
    
    # Clustering of embeddings - REDUCED CLUSTERS
    if is_train:
        # Clean embeddings before clustering
        embeddings_clean = np.copy(embeddings)
        embeddings_clean = np.nan_to_num(embeddings_clean, nan=0.0, posinf=0.0, neginf=0.0)
        
        kmeans = KMeans(n_clusters=20, random_state=42, n_init=5, max_iter=100)  # Reduced from 50 clusters
        features['emb_cluster'] = kmeans.fit_predict(embeddings_clean)
        CLUSTERS['emb_kmeans'] = kmeans
    else:
        kmeans = CLUSTERS.get('emb_kmeans')
        if kmeans is not None:
            embeddings_clean = np.copy(embeddings)
            embeddings_clean = np.nan_to_num(embeddings_clean, nan=0.0, posinf=0.0, neginf=0.0)
            features['emb_cluster'] = kmeans.predict(embeddings_clean)
        else:
            features['emb_cluster'] = 0
    
    # =========================
    # INTERACTION FEATURES
    # =========================
    features['ipq_x_qty'] = features['ipq'] * features['total_qty']
    features['ipq_x_qty_log'] = np.log1p(features['ipq_x_qty'])
    features['ipq_x_volume'] = features['ipq'] * features['total_volume']
    features['ipq_x_weight'] = features['ipq'] * features['total_weight']
    
    features['brand_x_quality'] = features['brand_tier'] * features['quality_score']
    features['category_x_premium'] = features['category_count'] * features['premium_word_count']
    features['category_x_quality'] = features['category_count'] * features['quality_score']
    
    features['text_length_x_ipq'] = features['text_length'] * features['ipq']
    features['word_count_x_quality'] = features['word_count'] * features['quality_score']
    features['word_count_x_premium'] = features['word_count'] * features['premium_word_count']
    
    features['tech_x_brand'] = features['is_tech_any'] * features['brand_tier']
    features['fashion_x_brand'] = features['is_fashion_any'] * features['brand_tier']
    features['consumable_x_ipq'] = features['is_consumable'] * features['ipq']
    
    features['qty_x_quality'] = features['total_qty'] * features['quality_score']
    features['material_x_premium'] = features['is_premium_material'] * features['has_premium']
    
    # =========================
    # RATIO FEATURES
    # =========================
    features['digit_to_word_ratio'] = features['digit_count'] / (features['word_count'] + 1)
    features['special_to_total_ratio'] = features['special_char_count'] / (features['text_length'] + 1)
    features['upper_to_total_ratio'] = features['upper_count'] / (features['text_length'] + 1)
    features['number_to_word_ratio'] = features['num_count'] / (features['word_count'] + 1)
    features['premium_to_total_ratio'] = features['premium_word_count'] / (features['word_count'] + 1)
    features['quality_to_total_ratio'] = features['quality_word_count'] / (features['word_count'] + 1)
    
    # =========================
    # POLYNOMIAL FEATURES (Selected)
    # =========================
    features['ipq_qty_poly'] = features['ipq'] * features['total_qty'] * features['quality_score']
    features['brand_cat_qual_poly'] = features['brand_tier'] * features['category_count'] * features['quality_score']
    
    # Convert to DataFrame
    features_df = pd.DataFrame(features)
    
    # =========================
    # HANDLE MISSING VALUES
    # =========================
    # Fill NaN/inf values before any further processing
    features_df = features_df.replace([np.inf, -np.inf], np.nan)
    features_df = features_df.fillna(0)
    
    # =========================
    # TARGET ENCODING (K-Fold) - OPTIMIZED
    # =========================
    categorical_cols = [col for col in features_df.columns if col.startswith('is_') or col.endswith('_count') or col in ['emb_cluster']]
    
    if is_train and prices is not None:
        log_prices = np.log1p(prices)
        global_mean = log_prices.mean()
        TARGET_ENCODINGS['global_mean'] = global_mean
        
        # Simple k-fold (no stratification for speed)
        kf_simple = KFold(n_splits=3, shuffle=True, random_state=42)  # Reduced from 5
        
        # Only encode top 20 most important categorical features
        for col in categorical_cols[:20]:  # Reduced from 30
            try:
                encoded_values = np.zeros(len(features_df))
                
                for train_idx, val_idx in kf_simple.split(features_df):
                    train_encoding = pd.DataFrame({
                        'cat': features_df[col].iloc[train_idx],
                        'price': log_prices[train_idx]
                    }).groupby('cat')['price'].mean().to_dict()
                    
                    # Simple encoding (no Bayesian smoothing for speed)
                    encoded_values[val_idx] = features_df[col].iloc[val_idx].map(train_encoding).fillna(global_mean)
                
                features_df[f'{col}_target_enc'] = encoded_values
                
                # Store full encoding
                full_encoding = pd.DataFrame({
                    'cat': features_df[col],
                    'price': log_prices
                }).groupby('cat')['price'].mean().to_dict()
                TARGET_ENCODINGS[col] = full_encoding
                
            except Exception as e:
                TARGET_ENCODINGS[col] = {}
                features_df[f'{col}_target_enc'] = global_mean
    else:
        global_mean = TARGET_ENCODINGS.get('global_mean', 0)
        for col in categorical_cols[:20]:
            encoding = TARGET_ENCODINGS.get(col, {})
            features_df[f'{col}_target_enc'] = features_df[col].map(encoding).fillna(global_mean)
    
    # =========================
    # SCALING
    # =========================
    scale_cols = ['text_length', 'word_count', 'total_qty', 'max_number', 'sum_numbers', 
                  'total_volume', 'total_weight', 'ipq', 'quality_score']
    
    if is_train:
        scaler = RobustScaler()
        features_df[scale_cols] = scaler.fit_transform(features_df[scale_cols])
        SCALERS['robust'] = scaler
        
        # Quantile transform for heavy-tailed features
        qt = QuantileTransformer(output_distribution='normal', random_state=42)
        qt_cols = ['max_number', 'sum_numbers', 'total_qty']
        features_df[[f'{c}_qt' for c in qt_cols]] = qt.fit_transform(features_df[qt_cols])
        SCALERS['quantile'] = qt
    else:
        scaler = SCALERS.get('robust')
        if scaler is not None:
            features_df[scale_cols] = scaler.transform(features_df[scale_cols])
        
        qt = SCALERS.get('quantile')
        if qt is not None:
            qt_cols = ['max_number', 'sum_numbers', 'total_qty']
            features_df[[f'{c}_qt' for c in qt_cols]] = qt.transform(features_df[qt_cols])
    
    # =========================
    # COMBINE WITH EMBEDDINGS
    # =========================
    X_combined = np.hstack([embeddings, features_df.values])
    
    print(f"Total features: {X_combined.shape[1]}")
    return X_combined

# ============================================================================
# MODEL TRAINING WITH ADVANCED TECHNIQUES
# ============================================================================

def train_ultra_ensemble():
    """Ultra-optimized ensemble with pseudo-labeling"""
    
    print("=" * 80)
    print("LOADING DATA")
    print("=" * 80)
    
    SAVE_DIR = Path("embeddings")
    X_train_emb = np.load(SAVE_DIR / "final_X_train.npy", allow_pickle=True).astype(np.float32)
    X_test_emb = np.load(SAVE_DIR / "final_X_test.npy", allow_pickle=True).astype(np.float32)
    
    df_train = pd.read_csv("train.csv")
    df_test = pd.read_csv("test.csv")
    
    print(f"Train: {df_train.shape}, Test: {df_test.shape}")
    
    # Feature extraction
    print("\n" + "=" * 80)
    print("FEATURE ENGINEERING")
    print("=" * 80)
    
    X_train = extract_ultra_features(df_train, X_train_emb, df_train['price'].values, is_train=True)
    X_test = extract_ultra_features(df_test, X_test_emb, is_train=False)
    
    y_train = np.log1p(df_train['price'].values)
    
    # Setup cross-validation
    kf = KFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.SEED)
    
    # Storage
    oof_lgb = np.zeros(len(X_train))
    oof_xgb = np.zeros(len(X_train))
    oof_cat = np.zeros(len(X_train))
    
    test_lgb = np.zeros(len(X_test))
    test_xgb = np.zeros(len(X_test))
    test_cat = np.zeros(len(X_test))
    
    fold_scores = []
    best_iterations = {'lgb': [], 'xgb': [], 'cat': []}
    
    print("\n" + "=" * 80)
    print("TRAINING ENSEMBLE MODELS")
    print("=" * 80)
    
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"\n{'='*80}")
        print(f"FOLD {fold_idx + 1}/{Config.N_SPLITS}")
        print(f"{'='*80}")
        
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # ============================
        # MODEL 1: LightGBM
        # ============================
        print(f"\n[Fold {fold_idx + 1}] Training LightGBM...")
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        lgb_model = lgb.train(
            Config.LGB_PARAMS,
            train_data,
            num_boost_round=5000,
            valid_sets=[val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=150),
                lgb.log_evaluation(period=0)
            ]
        )
        
        oof_lgb[val_idx] = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
        test_lgb += lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration) / Config.N_SPLITS
        best_iterations['lgb'].append(lgb_model.best_iteration)
        
        # ============================
        # MODEL 2: XGBoost
        # ============================
        print(f"[Fold {fold_idx + 1}] Training XGBoost...")
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        dtest = xgb.DMatrix(X_test)
        
        xgb_model = xgb.train(
            Config.XGB_PARAMS,
            dtrain,
            num_boost_round=5000,
            evals=[(dval, 'eval')],
            early_stopping_rounds=150,
            verbose_eval=False
        )
        
        oof_xgb[val_idx] = xgb_model.predict(dval)
        test_xgb += xgb_model.predict(dtest) / Config.N_SPLITS
        best_iterations['xgb'].append(xgb_model.best_iteration)
        
        # ============================
        # MODEL 3: CatBoost
        # ============================
        print(f"[Fold {fold_idx + 1}] Training CatBoost...")
        
        cat_model = CatBoostRegressor(**Config.CAT_PARAMS)
        cat_model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            verbose=False
        )
        
        oof_cat[val_idx] = cat_model.predict(X_val)
        test_cat += cat_model.predict(X_test) / Config.N_SPLITS
        best_iterations['cat'].append(cat_model.best_iteration_)
        
        # ============================
        # FOLD EVALUATION
        # ============================
        # Weighted ensemble (CatBoost gets more weight)
        oof_ensemble = (oof_lgb[val_idx] * 0.3 + oof_xgb[val_idx] * 0.3 + oof_cat[val_idx] * 0.4)
        
        y_val_actual = np.expm1(y_val)
        y_pred_actual = np.expm1(oof_ensemble)
        
        # SMAPE calculation
        denominator = (np.abs(y_val_actual) + np.abs(y_pred_actual)) / 2.0
        diff = np.abs(y_val_actual - y_pred_actual)
        smape = np.mean(np.where(denominator == 0, 0, diff / denominator)) * 100
        
        fold_scores.append(smape)
        print(f"\nFold {fold_idx + 1} SMAPE: {smape:.4f}%")
        print(f"  LightGBM: {best_iterations['lgb'][-1]} iterations")
        print(f"  XGBoost:  {best_iterations['xgb'][-1]} iterations")
        print(f"  CatBoost: {best_iterations['cat'][-1]} iterations")
    
    # ============================
    # META-LEARNER (Stacking)
    # ============================
    print("\n" + "=" * 80)
    print("TRAINING META-LEARNER")
    print("=" * 80)
    
    # Create stacking features
    oof_stack = np.column_stack([
        oof_lgb,
        oof_xgb,
        oof_cat,
        (oof_lgb + oof_xgb) / 2,  # Pairwise averages
        (oof_lgb + oof_cat) / 2,
        (oof_xgb + oof_cat) / 2,
        (oof_lgb + oof_xgb + oof_cat) / 3,  # Simple average
    ])
    
    test_stack = np.column_stack([
        test_lgb,
        test_xgb,
        test_cat,
        (test_lgb + test_xgb) / 2,
        (test_lgb + test_cat) / 2,
        (test_xgb + test_cat) / 2,
        (test_lgb + test_xgb + test_cat) / 3,
    ])
    
    # Ridge meta-learner with cross-validation
    meta_oof = np.zeros(len(X_train))
    meta_test = np.zeros(len(X_test))
    
    for train_idx, val_idx in kf.split(X_train):
        meta_model = Ridge(alpha=0.5, random_state=Config.SEED)
        meta_model.fit(oof_stack[train_idx], y_train[train_idx])
        meta_oof[val_idx] = meta_model.predict(oof_stack[val_idx])
        meta_test += meta_model.predict(test_stack) / Config.N_SPLITS
    
    # ============================
    # PSEUDO-LABELING (Optional - Careful!)
    # ============================
    # Only use high-confidence predictions
    test_pred_variance = np.std([test_lgb, test_xgb, test_cat], axis=0)
    high_confidence_mask = test_pred_variance < np.percentile(test_pred_variance, 10)
    
    print(f"\nHigh confidence test samples: {high_confidence_mask.sum()}")
    
    # ============================
    # FINAL PREDICTION
    # ============================
    test_pred_log = meta_test
    test_pred_final = np.expm1(test_pred_log)
    
    # Post-processing
    test_pred_final = np.maximum(test_pred_final, 0.01)
    
    # Clip extreme outliers
    lower_bound = np.percentile(np.expm1(y_train), 0.1)
    upper_bound = np.percentile(np.expm1(y_train), 99.9)
    test_pred_final = np.clip(test_pred_final, lower_bound * 0.5, upper_bound * 1.5)
    
    # ============================
    # RESULTS
    # ============================
    print("\n" + "=" * 80)
    print("FINAL RESULTS")
    print("=" * 80)
    print(f"Mean CV SMAPE: {np.mean(fold_scores):.4f}% (+/- {np.std(fold_scores):.4f}%)")
    print(f"Fold scores: {[f'{s:.4f}%' for s in fold_scores]}")
    print(f"\nBest iterations (avg):")
    print(f"  LightGBM: {np.mean(best_iterations['lgb']):.0f}")
    print(f"  XGBoost:  {np.mean(best_iterations['xgb']):.0f}")
    print(f"  CatBoost: {np.mean(best_iterations['cat']):.0f}")
    
    # Calculate meta SMAPE
    y_train_actual = np.expm1(y_train)
    meta_oof_actual = np.expm1(meta_oof)
    denominator = (np.abs(y_train_actual) + np.abs(meta_oof_actual)) / 2.0
    diff = np.abs(y_train_actual - meta_oof_actual)
    meta_smape = np.mean(np.where(denominator == 0, 0, diff / denominator)) * 100
    print(f"\nMeta-learner OOF SMAPE: {meta_smape:.4f}%")
    
    # Create submission
    submission = pd.DataFrame({
        'sample_id': df_test['sample_id'].values,
        'price': test_pred_final
    })
    
    submission.to_csv('submission_ultra_ensemble.csv', index=False)
    
    print("\n" + "=" * 80)
    print("SUBMISSION CREATED")
    print("=" * 80)
    print(f"✓ File: submission_ultra_ensemble.csv")
    print(f"✓ Shape: {submission.shape}")
    print(f"\nPrice Statistics:")
    print(f"  Min:    ${submission['price'].min():.2f}")
    print(f"  Max:    ${submission['price'].max():.2f}")
    print(f"  Mean:   ${submission['price'].mean():.2f}")
    print(f"  Median: ${submission['price'].median():.2f}")
    print(f"  Std:    ${submission['price'].std():.2f}")
    print(f"  Q25:    ${submission['price'].quantile(0.25):.2f}")
    print(f"  Q75:    ${submission['price'].quantile(0.75):.2f}")
    
    # Individual model submissions (for analysis)
    pd.DataFrame({
        'sample_id': df_test['sample_id'].values,
        'price': np.expm1(test_lgb)
    }).to_csv('submission_lgb_only.csv', index=False)
    
    pd.DataFrame({
        'sample_id': df_test['sample_id'].values,
        'price': np.expm1(test_xgb)
    }).to_csv('submission_xgb_only.csv', index=False)
    
    pd.DataFrame({
        'sample_id': df_test['sample_id'].values,
        'price': np.expm1(test_cat)
    }).to_csv('submission_cat_only.csv', index=False)
    
    print("\n✓ Individual model submissions also saved for comparison")
    
    return submission

# ============================================================================
# RUN
# ============================================================================

if __name__ == "__main__":
    np.random.seed(Config.SEED)
    submission = train_ultra_ensemble()

LOADING DATA
Train: (75000, 4), Test: (75000, 3)

FEATURE ENGINEERING
Extracting features for 75000 samples...
Total features: 1266
Extracting features for 75000 samples...
Total features: 1266

TRAINING ENSEMBLE MODELS

FOLD 1/5

[Fold 1] Training LightGBM...
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[3309]	valid_0's rmse: 0.703785
[Fold 1] Training XGBoost...
[Fold 1] Training CatBoost...

Fold 1 SMAPE: 53.9438%
  LightGBM: 3309 iterations
  XGBoost:  3296 iterations
  CatBoost: 1996 iterations

FOLD 2/5

[Fold 2] Training LightGBM...
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[3821]	valid_0's rmse: 0.691565
[Fold 2] Training XGBoost...
[Fold 2] Training CatBoost...

Fold 2 SMAPE: 53.2989%
  LightGBM: 3821 iterations
  XGBoost:  4103 iterations
  CatBoost: 1997 iterations

FOLD 3/5

[Fold 3] Training LightGBM...
Training until validation scores don't improve for 150 rounds
Ear

Claude Ultimate

In [3]:
# ===================================================================
# ENHANCED MLP FUSION WITH ADVANCED FEATURE ENGINEERING
# Target: SMAPE < 45% with Maximum Data Extraction
# ===================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from tqdm import tqdm
import gc
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 ENHANCED MLP FUSION: Maximum Feature Extraction")
print("="*70)

# ===================================================================
# ADVANCED FEATURE ENGINEERING
# ===================================================================
def extract_advanced_features(df, is_train=True):
    """Extract rich features from catalog_content and other columns"""
    print(f"  Extracting advanced features...")
    
    features = {}
    
    # Text length features
    features['title_len'] = df['catalog_content'].str.len()
    features['word_count'] = df['catalog_content'].str.split().str.len()
    features['avg_word_len'] = features['title_len'] / (features['word_count'] + 1)
    
    # Brand features (if present)
    if 'brand' in df.columns:
        # Brand frequency encoding
        brand_freq = df['brand'].value_counts()
        features['brand_freq'] = df['brand'].map(brand_freq).fillna(0)
        
        # Brand mean price (only for train)
        if is_train and 'price' in df.columns:
            brand_mean = df.groupby('brand')['price'].mean()
            features['brand_mean_price'] = df['brand'].map(brand_mean).fillna(df['price'].median())
    
    # Quantity features
    if 'quantity' in df.columns:
        features['quantity'] = df['quantity'].fillna(1)
        features['log_quantity'] = np.log1p(features['quantity'])
        features['quantity_squared'] = features['quantity'] ** 2
    
    # Text pattern features
    features['has_digits'] = df['catalog_content'].str.contains(r'\d', regex=True).astype(int)
    features['has_special_chars'] = df['catalog_content'].str.contains(r'[^a-zA-Z0-9\s]', regex=True).astype(int)
    features['uppercase_ratio'] = df['catalog_content'].str.count(r'[A-Z]') / (features['title_len'] + 1)
    
    # Extract numeric values from text
    features['num_numbers'] = df['catalog_content'].str.findall(r'\d+').str.len().fillna(0)
    
    # Check for common price indicators
    price_keywords = ['premium', 'luxury', 'pro', 'plus', 'max', 'ultra', 'deluxe']
    budget_keywords = ['basic', 'mini', 'lite', 'eco', 'value']
    
    features['has_premium_word'] = df['catalog_content'].str.lower().str.contains('|'.join(price_keywords)).astype(int)
    features['has_budget_word'] = df['catalog_content'].str.lower().str.contains('|'.join(budget_keywords)).astype(int)
    
    return pd.DataFrame(features)

# ===================================================================
# ENHANCED MLP ARCHITECTURE WITH RESIDUAL CONNECTIONS
# ===================================================================
class EnhancedMultimodalFusionMLP(nn.Module):
    """
    Enhanced fusion with:
    - Residual connections
    - Better attention mechanism
    - Gated fusion
    - More sophisticated architecture
    """
    def __init__(self, text_dim, image_dim, other_dim, hidden_dim=768, dropout=0.25):
        super().__init__()
        
        # Individual encoders with residual capability
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.other_encoder = nn.Sequential(
            nn.Linear(other_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.GELU()
        )
        
        # Cross-attention between modalities
        self.text_to_image_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        self.image_to_text_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        
        # Gated fusion mechanism
        self.gate = nn.Sequential(
        nn.Linear(hidden_dim * 2 + 128, hidden_dim * 2),  # Changed from hidden_dim to hidden_dim * 2
        nn.Sigmoid()
        )
        
        # Main fusion network with residual connections
        fusion_input_dim = hidden_dim * 2 + 128
        self.fusion1 = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.fusion2 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.fusion3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        
        self.output = nn.Linear(hidden_dim // 2, 1)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, text_emb, image_emb, other_emb):
        # Encode each modality
        text_enc = self.text_encoder(text_emb)
        image_enc = self.image_encoder(image_emb)
        other_enc = self.other_encoder(other_emb)
        
        # Bidirectional cross-attention
        text_att, _ = self.text_to_image_attn(
            text_enc.unsqueeze(1), 
            image_enc.unsqueeze(1), 
            image_enc.unsqueeze(1)
        )
        text_att = text_att.squeeze(1)
        
        image_att, _ = self.image_to_text_attn(
            image_enc.unsqueeze(1), 
            text_enc.unsqueeze(1), 
            text_enc.unsqueeze(1)
        )
        image_att = image_att.squeeze(1)
        
        # Combine with residuals
        text_combined = text_enc + text_att
        image_combined = image_enc + image_att
        
        # Concatenate all features
        fused = torch.cat([text_combined, image_combined, other_enc], dim=1)
        
        # Gated fusion
        gate_values = self.gate(fused)
        
        # Apply fusion layers with residuals
        x = self.fusion1(fused)
        x = x * gate_values[:, :x.size(1)]  # Apply gating
        
        x = self.fusion2(x)
        x = self.fusion3(x)
        output = self.output(x)
        
        return output

# ===================================================================
# CUSTOM LOSS FUNCTION - SMAPE-inspired
# ===================================================================
def smape_loss(pred, target, epsilon=1e-3):
    """SMAPE-inspired loss that directly optimizes the evaluation metric"""
    # Work in log space to match our target
    pred_exp = torch.exp(pred)
    target_exp = torch.exp(target)
    
    numerator = torch.abs(pred_exp - target_exp)
    denominator = (torch.abs(target_exp) + torch.abs(pred_exp)) / 2.0 + epsilon
    
    return torch.mean(numerator / denominator)

def combined_loss(pred, target, alpha=0.5):
    """Combine SMAPE loss with Huber loss for stability"""
    smape = smape_loss(pred, target)
    huber = nn.SmoothL1Loss()(pred, target)
    return alpha * smape + (1 - alpha) * huber

# ===================================================================
# TRAINING FUNCTION WITH ADVANCED TECHNIQUES
# ===================================================================
def train_enhanced_mlp(X_text_tr, X_image_tr, X_other_tr, y_tr, 
                       X_text_val, X_image_val, X_other_val, y_val,
                       epochs=150, batch_size=256, lr=3e-4):
    
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"  Using device: {device}")
    
    text_dim = X_text_tr.shape[1]
    image_dim = X_image_tr.shape[1]
    other_dim = X_other_tr.shape[1]
    
    model = EnhancedMultimodalFusionMLP(text_dim, image_dim, other_dim).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-5, betas=(0.9, 0.999))
    
    # Cosine annealing with warm restarts
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=15, T_mult=2, eta_min=1e-6
    )
    
    # Create datasets
    train_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_text_tr),
        torch.FloatTensor(X_image_tr),
        torch.FloatTensor(X_other_tr),
        torch.FloatTensor(y_tr).unsqueeze(1)
    )
    
    val_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_text_val),
        torch.FloatTensor(X_image_val),
        torch.FloatTensor(X_other_val),
        torch.FloatTensor(y_val).unsqueeze(1)
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    best_val_loss = float('inf')
    patience = 20
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        
        for text_b, image_b, other_b, y_b in train_loader:
            text_b = text_b.to(device)
            image_b = image_b.to(device)
            other_b = other_b.to(device)
            y_b = y_b.to(device)
            
            optimizer.zero_grad()
            output = model(text_b, image_b, other_b)
            loss = combined_loss(output, y_b)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for text_b, image_b, other_b, y_b in val_loader:
                text_b = text_b.to(device)
                image_b = image_b.to(device)
                other_b = other_b.to(device)
                y_b = y_b.to(device)
                
                output = model(text_b, image_b, other_b)
                val_loss += combined_loss(output, y_b).item()
        
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        scheduler.step()
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1}")
            break
        
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}: train_loss={train_loss:.5f}, val_loss={val_loss:.5f}")
    
    model.load_state_dict(best_model_state)
    return model

# ===================================================================
# PREDICTION FUNCTION
# ===================================================================
def predict_enhanced(model, X_text, X_image, X_other, batch_size=512):
    device = next(model.parameters()).device
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(X_text), batch_size), desc="Predicting", leave=False):
            end_idx = min(i + batch_size, len(X_text))
            
            text_b = torch.FloatTensor(X_text[i:end_idx]).to(device)
            image_b = torch.FloatTensor(X_image[i:end_idx]).to(device)
            other_b = torch.FloatTensor(X_other[i:end_idx]).to(device)
            
            output = model(text_b, image_b, other_b)
            predictions.append(output.cpu().numpy())
    
    return np.vstack(predictions).flatten()

# ===================================================================
# MAIN EXECUTION
# ===================================================================
print("\n[1/5] Loading data and embeddings...")
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Load embeddings
X_train_full = np.load("final_X_train_medium_with_brand.npy", allow_pickle=False)
X_test_full = np.load("final_X_test_medium_with_brand.npy", allow_pickle=False)

# Define dimensions
text_dim = 384
image_dim = 512

# Slice embeddings
train_text = X_train_full[:, :text_dim]
train_image = X_train_full[:, text_dim:text_dim+image_dim]
train_other_base = X_train_full[:, text_dim+image_dim:]

test_text = X_test_full[:, :text_dim]
test_image = X_test_full[:, text_dim:text_dim+image_dim]
test_other_base = X_test_full[:, text_dim+image_dim:]

print(f"✓ Loaded embeddings")
del X_train_full, X_test_full
gc.collect()

# ===================================================================
# ADVANCED FEATURE ENGINEERING
# ===================================================================
print("\n[2/5] Engineering advanced features...")
train_extra_features = extract_advanced_features(df_train, is_train=True)
test_extra_features = extract_advanced_features(df_test, is_train=False)

# Combine with existing other features
train_other = np.hstack([train_other_base, train_extra_features.values])
test_other = np.hstack([test_other_base, test_extra_features.values])

print(f"✓ Enhanced features: {train_other.shape[1]} dimensions")
del train_other_base, test_other_base
gc.collect()

# Target transformation
y_train_log = np.log1p(df_train['price'].values)

# ===================================================================
# ADVANCED SCALING
# ===================================================================
print("\n[3/5] Applying robust scaling...")

# Use QuantileTransformer for embeddings (more robust to outliers)
text_scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal')
image_scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal')
other_scaler = RobustScaler()

train_text_scaled = text_scaler.fit_transform(train_text)
test_text_scaled = text_scaler.transform(test_text)

train_image_scaled = image_scaler.fit_transform(train_image)
test_image_scaled = image_scaler.transform(test_image)

train_other_scaled = other_scaler.fit_transform(train_other)
test_other_scaled = other_scaler.transform(test_other)

print("✓ Scaling complete")
del train_text, train_image, train_other, test_text, test_image, test_other
gc.collect()

# ===================================================================
# K-FOLD CROSS-VALIDATION WITH STRATIFICATION
# ===================================================================
print("\n[4/5] Training with K-Fold CV...")

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_text_scaled))
test_preds = np.zeros(len(test_text_scaled))

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_text_scaled), 1):
    print(f"\n{'─'*70}")
    print(f"📊 FOLD {fold}/{N_FOLDS}")
    print(f"{'─'*70}")
    
    model = train_enhanced_mlp(
        train_text_scaled[train_idx], 
        train_image_scaled[train_idx], 
        train_other_scaled[train_idx], 
        y_train_log[train_idx],
        train_text_scaled[val_idx], 
        train_image_scaled[val_idx], 
        train_other_scaled[val_idx], 
        y_train_log[val_idx]
    )
    
    # OOF predictions
    oof_preds[val_idx] = predict_enhanced(
        model, 
        train_text_scaled[val_idx], 
        train_image_scaled[val_idx], 
        train_other_scaled[val_idx]
    )
    
    # Test predictions
    fold_test_preds = predict_enhanced(
        model, 
        test_text_scaled, 
        test_image_scaled, 
        test_other_scaled
    )
    test_preds += fold_test_preds / N_FOLDS
    
    # Calculate fold SMAPE
    val_pred_price = np.expm1(oof_preds[val_idx])
    val_actual_price = np.expm1(y_train_log[val_idx])
    
    fold_smape = np.mean(
        2 * np.abs(val_pred_price - val_actual_price) / 
        (np.abs(val_actual_price) + np.abs(val_pred_price) + 1e-8)
    ) * 100
    
    fold_scores.append(fold_smape)
    print(f"  📈 Fold {fold} SMAPE: {fold_smape:.4f}%")
    
    del model
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

# ===================================================================
# FINAL EVALUATION
# ===================================================================
print("\n[5/5] Final evaluation and submission...")

oof_prices = np.expm1(oof_preds)
actual_prices = df_train['price'].values

overall_smape = np.mean(
    2 * np.abs(oof_prices - actual_prices) / 
    (np.abs(actual_prices) + np.abs(oof_prices) + 1e-8)
) * 100

print("\n" + "="*70)
print(f"📊 CROSS-VALIDATION RESULTS")
print("="*70)
for i, score in enumerate(fold_scores, 1):
    print(f"  Fold {i}: {score:.4f}%")
print(f"\n  Mean: {np.mean(fold_scores):.4f}%")
print(f"  Std:  {np.std(fold_scores):.4f}%")
print("\n" + "="*70)
print(f"🎯 FINAL OOF SMAPE: {overall_smape:.4f}%")
print("="*70)

# ===================================================================
# CREATE SUBMISSION
# ===================================================================
final_predictions = np.expm1(test_preds)
final_predictions = np.clip(final_predictions, 0.01, None)

submission = pd.DataFrame({
    'sample_id': df_test['sample_id'],
    'price': final_predictions
})

submission.to_csv('enhanced_mlp_fusion_submission.csv', index=False)

print("\n✅ Submission saved: enhanced_mlp_fusion_submission.csv")
print("\n📋 Prediction statistics:")
print(f"  Min:    ${final_predictions.min():.2f}")
print(f"  Max:    ${final_predictions.max():.2f}")
print(f"  Mean:   ${final_predictions.mean():.2f}")
print(f"  Median: ${np.median(final_predictions):.2f}")
print("\n" + "="*70)

🚀 ENHANCED MLP FUSION: Maximum Feature Extraction

[1/5] Loading data and embeddings...
✓ Loaded embeddings

[2/5] Engineering advanced features...
  Extracting advanced features...
  Extracting advanced features...
✓ Enhanced features: 165 dimensions

[3/5] Applying robust scaling...
✓ Scaling complete

[4/5] Training with K-Fold CV...

──────────────────────────────────────────────────────────────────────
📊 FOLD 1/5
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25115, val_loss=0.33332
    Epoch 20: train_loss=0.22530, val_loss=0.33031
    Epoch 30: train_loss=0.16742, val_loss=0.32765
    Epoch 40: train_loss=0.13476, val_loss=0.32228
    Epoch 50: train_loss=0.15560, val_loss=0.33483
    Early stopping at epoch 57


  📈 Fold 1 SMAPE: 47.9270%

──────────────────────────────────────────────────────────────────────
📊 FOLD 2/5
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25089, val_loss=0.32996
    Epoch 20: train_loss=0.22590, val_loss=0.33083
    Epoch 30: train_loss=0.16744, val_loss=0.32861
    Epoch 40: train_loss=0.13493, val_loss=0.32276
    Epoch 50: train_loss=0.15661, val_loss=0.32881
    Early stopping at epoch 59


  📈 Fold 2 SMAPE: 48.0331%

──────────────────────────────────────────────────────────────────────
📊 FOLD 3/5
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25125, val_loss=0.33338
    Epoch 20: train_loss=0.22566, val_loss=0.34064
    Epoch 30: train_loss=0.16783, val_loss=0.32718
    Epoch 40: train_loss=0.13532, val_loss=0.32223
    Epoch 50: train_loss=0.15581, val_loss=0.33189
    Early stopping at epoch 58


  📈 Fold 3 SMAPE: 47.6499%

──────────────────────────────────────────────────────────────────────
📊 FOLD 4/5
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25245, val_loss=0.32279
    Epoch 20: train_loss=0.22778, val_loss=0.33445
    Epoch 30: train_loss=0.16844, val_loss=0.32128
    Epoch 40: train_loss=0.13573, val_loss=0.31771
    Epoch 50: train_loss=0.15690, val_loss=0.32375
    Epoch 60: train_loss=0.13162, val_loss=0.31818
    Early stopping at epoch 63


  📈 Fold 4 SMAPE: 47.0392%

──────────────────────────────────────────────────────────────────────
📊 FOLD 5/5
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25212, val_loss=0.32844
    Epoch 20: train_loss=0.22598, val_loss=0.32957
    Epoch 30: train_loss=0.16846, val_loss=0.32765
    Epoch 40: train_loss=0.13479, val_loss=0.31869
    Epoch 50: train_loss=0.15509, val_loss=0.32508
    Early stopping at epoch 58


  📈 Fold 5 SMAPE: 47.3149%

[5/5] Final evaluation and submission...

📊 CROSS-VALIDATION RESULTS
  Fold 1: 47.9270%
  Fold 2: 48.0331%
  Fold 3: 47.6499%
  Fold 4: 47.0392%
  Fold 5: 47.3149%

  Mean: 47.5928%
  Std:  0.3722%

🎯 FINAL OOF SMAPE: 47.5928%

✅ Submission saved: enhanced_mlp_fusion_submission.csv

📋 Prediction statistics:
  Min:    $0.39
  Max:    $357.65
  Mean:   $20.08
  Median: $13.52



claude ultimate 2

In [6]:
# ===================================================================
# ENHANCED MLP FUSION WITH ADVANCED FEATURE ENGINEERING
# Target: SMAPE < 45% with Maximum Data Extraction
# ===================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from tqdm import tqdm
import gc
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 ENHANCED MLP FUSION: Maximum Feature Extraction")
print("="*70)

# ===================================================================
# ADVANCED FEATURE ENGINEERING
# ===================================================================
def extract_advanced_features(df, is_train=True):
    """Extract rich features from catalog_content and other columns"""
    print(f"  Extracting advanced features...")
    
    features = {}
    
    # Text length features
    features['title_len'] = df['catalog_content'].str.len()
    features['word_count'] = df['catalog_content'].str.split().str.len()
    features['avg_word_len'] = features['title_len'] / (features['word_count'] + 1)
    
    # Brand features (if present)
    if 'brand' in df.columns:
        # Brand frequency encoding
        brand_freq = df['brand'].value_counts()
        features['brand_freq'] = df['brand'].map(brand_freq).fillna(0)
        
        # Brand mean price (only for train)
        if is_train and 'price' in df.columns:
            brand_mean = df.groupby('brand')['price'].mean()
            features['brand_mean_price'] = df['brand'].map(brand_mean).fillna(df['price'].median())
    
    # Quantity features
    if 'quantity' in df.columns:
        features['quantity'] = df['quantity'].fillna(1)
        features['log_quantity'] = np.log1p(features['quantity'])
        features['quantity_squared'] = features['quantity'] ** 2
    
    # Text pattern features
    features['has_digits'] = df['catalog_content'].str.contains(r'\d', regex=True).astype(int)
    features['has_special_chars'] = df['catalog_content'].str.contains(r'[^a-zA-Z0-9\s]', regex=True).astype(int)
    features['uppercase_ratio'] = df['catalog_content'].str.count(r'[A-Z]') / (features['title_len'] + 1)
    
    # Extract numeric values from text
    features['num_numbers'] = df['catalog_content'].str.findall(r'\d+').str.len().fillna(0)
    
    # Check for common price indicators
    price_keywords = ['premium', 'luxury', 'pro', 'plus', 'max', 'ultra', 'deluxe']
    budget_keywords = ['basic', 'mini', 'lite', 'eco', 'value']
    
    features['has_premium_word'] = df['catalog_content'].str.lower().str.contains('|'.join(price_keywords)).astype(int)
    features['has_budget_word'] = df['catalog_content'].str.lower().str.contains('|'.join(budget_keywords)).astype(int)
    
    return pd.DataFrame(features)

# ===================================================================
# ENHANCED MLP ARCHITECTURE WITH RESIDUAL CONNECTIONS
# ===================================================================
class EnhancedMultimodalFusionMLP(nn.Module):
    """
    Enhanced fusion with:
    - Residual connections
    - Better attention mechanism
    - Gated fusion
    - More sophisticated architecture
    """
    def __init__(self, text_dim, image_dim, other_dim, hidden_dim=768, dropout=0.25):
        super().__init__()
        
        # Individual encoders with residual capability
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.other_encoder = nn.Sequential(
            nn.Linear(other_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.GELU()
        )
        
        # Cross-attention between modalities
        self.text_to_image_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        self.image_to_text_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        
        # Gated fusion mechanism
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 128, hidden_dim * 2),  # Changed from hidden_dim to hidden_dim * 2
            nn.Sigmoid()
        )
        
        # Main fusion network with residual connections
        fusion_input_dim = hidden_dim * 2 + 128
        self.fusion1 = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.fusion2 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.fusion3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        
        self.output = nn.Linear(hidden_dim // 2, 1)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, text_emb, image_emb, other_emb):
        # Encode each modality
        text_enc = self.text_encoder(text_emb)
        image_enc = self.image_encoder(image_emb)
        other_enc = self.other_encoder(other_emb)
        
        # Bidirectional cross-attention
        text_att, _ = self.text_to_image_attn(
            text_enc.unsqueeze(1), 
            image_enc.unsqueeze(1), 
            image_enc.unsqueeze(1)
        )
        text_att = text_att.squeeze(1)
        
        image_att, _ = self.image_to_text_attn(
            image_enc.unsqueeze(1), 
            text_enc.unsqueeze(1), 
            text_enc.unsqueeze(1)
        )
        image_att = image_att.squeeze(1)
        
        # Combine with residuals
        text_combined = text_enc + text_att
        image_combined = image_enc + image_att
        
        # Concatenate all features
        fused = torch.cat([text_combined, image_combined, other_enc], dim=1)
        
        # Gated fusion
        gate_values = self.gate(fused)
        
        # Apply fusion layers with residuals
        x = self.fusion1(fused)
        x = x * gate_values[:, :x.size(1)]  # Apply gating
        
        x = self.fusion2(x)
        x = self.fusion3(x)
        output = self.output(x)
        
        return output

# ===================================================================
# CUSTOM LOSS FUNCTION - SMAPE-inspired
# ===================================================================
def smape_loss(pred, target, epsilon=1e-3):
    """SMAPE-inspired loss that directly optimizes the evaluation metric"""
    # Work in log space to match our target
    pred_exp = torch.exp(pred)
    target_exp = torch.exp(target)
    
    numerator = torch.abs(pred_exp - target_exp)
    denominator = (torch.abs(target_exp) + torch.abs(pred_exp)) / 2.0 + epsilon
    
    return torch.mean(numerator / denominator)

def combined_loss(pred, target, alpha=0.5):
    """Combine SMAPE loss with Huber loss for stability"""
    smape = smape_loss(pred, target)
    huber = nn.SmoothL1Loss()(pred, target)
    return alpha * smape + (1 - alpha) * huber

# ===================================================================
# TRAINING FUNCTION WITH ADVANCED TECHNIQUES
# ===================================================================
def train_enhanced_mlp(X_text_tr, X_image_tr, X_other_tr, y_tr, 
                       X_text_val, X_image_val, X_other_val, y_val,
                       epochs=150, batch_size=256, lr=3e-4):
    
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"  Using device: {device}")
    
    text_dim = X_text_tr.shape[1]
    image_dim = X_image_tr.shape[1]
    other_dim = X_other_tr.shape[1]
    
    model = EnhancedMultimodalFusionMLP(text_dim, image_dim, other_dim).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-5, betas=(0.9, 0.999))
    
    # Cosine annealing with warm restarts
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=15, T_mult=2, eta_min=1e-6
    )
    
    # Create datasets
    train_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_text_tr),
        torch.FloatTensor(X_image_tr),
        torch.FloatTensor(X_other_tr),
        torch.FloatTensor(y_tr).unsqueeze(1)
    )
    
    val_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_text_val),
        torch.FloatTensor(X_image_val),
        torch.FloatTensor(X_other_val),
        torch.FloatTensor(y_val).unsqueeze(1)
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    best_val_loss = float('inf')
    patience = 20
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        
        for text_b, image_b, other_b, y_b in train_loader:
            text_b = text_b.to(device)
            image_b = image_b.to(device)
            other_b = other_b.to(device)
            y_b = y_b.to(device)
            
            optimizer.zero_grad()
            output = model(text_b, image_b, other_b)
            loss = combined_loss(output, y_b)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for text_b, image_b, other_b, y_b in val_loader:
                text_b = text_b.to(device)
                image_b = image_b.to(device)
                other_b = other_b.to(device)
                y_b = y_b.to(device)
                
                output = model(text_b, image_b, other_b)
                val_loss += combined_loss(output, y_b).item()
        
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        scheduler.step()
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1}")
            break
        
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}: train_loss={train_loss:.5f}, val_loss={val_loss:.5f}")
    
    model.load_state_dict(best_model_state)
    return model

# ===================================================================
# PREDICTION FUNCTION
# ===================================================================
def predict_enhanced(model, X_text, X_image, X_other, batch_size=512):
    device = next(model.parameters()).device
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(X_text), batch_size), desc="Predicting", leave=False):
            end_idx = min(i + batch_size, len(X_text))
            
            text_b = torch.FloatTensor(X_text[i:end_idx]).to(device)
            image_b = torch.FloatTensor(X_image[i:end_idx]).to(device)
            other_b = torch.FloatTensor(X_other[i:end_idx]).to(device)
            
            output = model(text_b, image_b, other_b)
            predictions.append(output.cpu().numpy())
    
    return np.vstack(predictions).flatten()

# ===================================================================
# MAIN EXECUTION
# ===================================================================
print("\n[1/5] Loading data and embeddings...")
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Load embeddings
X_train_full = np.load("final_X_train_medium_with_brand.npy", allow_pickle=False)
X_test_full = np.load("final_X_test_medium_with_brand.npy", allow_pickle=False)

# Define dimensions
text_dim = 384
image_dim = 512

# Slice embeddings
train_text = X_train_full[:, :text_dim]
train_image = X_train_full[:, text_dim:text_dim+image_dim]
train_other_base = X_train_full[:, text_dim+image_dim:]

test_text = X_test_full[:, :text_dim]
test_image = X_test_full[:, text_dim:text_dim+image_dim]
test_other_base = X_test_full[:, text_dim+image_dim:]

print(f"✓ Loaded embeddings")
del X_train_full, X_test_full
gc.collect()

# ===================================================================
# ADVANCED FEATURE ENGINEERING
# ===================================================================
print("\n[2/5] Engineering advanced features...")
train_extra_features = extract_advanced_features(df_train, is_train=True)
test_extra_features = extract_advanced_features(df_test, is_train=False)

# Combine with existing other features
train_other = np.hstack([train_other_base, train_extra_features.values])
test_other = np.hstack([test_other_base, test_extra_features.values])

print(f"✓ Enhanced features: {train_other.shape[1]} dimensions")
del train_other_base, test_other_base
gc.collect()

# Target transformation
y_train_log = np.log1p(df_train['price'].values)

# ===================================================================
# ADVANCED SCALING
# ===================================================================
print("\n[3/5] Applying robust scaling...")

# Use QuantileTransformer for embeddings (more robust to outliers)
text_scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal')
image_scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal')
other_scaler = RobustScaler()

train_text_scaled = text_scaler.fit_transform(train_text)
test_text_scaled = text_scaler.transform(test_text)

train_image_scaled = image_scaler.fit_transform(train_image)
test_image_scaled = image_scaler.transform(test_image)

train_other_scaled = other_scaler.fit_transform(train_other)
test_other_scaled = other_scaler.transform(test_other)

print("✓ Scaling complete")
del train_text, train_image, train_other, test_text, test_image, test_other
gc.collect()

# ===================================================================
# K-FOLD CROSS-VALIDATION WITH STRATIFICATION
# ===================================================================
print("\n[4/5] Training with K-Fold CV...")

N_FOLDS = 7  # Increased from 5 to 7 for better generalization
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_text_scaled))
test_preds = np.zeros(len(test_text_scaled))

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_text_scaled), 1):
    print(f"\n{'─'*70}")
    print(f"📊 FOLD {fold}/{N_FOLDS}")
    print(f"{'─'*70}")
    
    model = train_enhanced_mlp(
        train_text_scaled[train_idx], 
        train_image_scaled[train_idx], 
        train_other_scaled[train_idx], 
        y_train_log[train_idx],
        train_text_scaled[val_idx], 
        train_image_scaled[val_idx], 
        train_other_scaled[val_idx], 
        y_train_log[val_idx]
    )
    
    # OOF predictions
    oof_preds[val_idx] = predict_enhanced(
        model, 
        train_text_scaled[val_idx], 
        train_image_scaled[val_idx], 
        train_other_scaled[val_idx]
    )
    
    # Test predictions
    fold_test_preds = predict_enhanced(
        model, 
        test_text_scaled, 
        test_image_scaled, 
        test_other_scaled
    )
    test_preds += fold_test_preds / N_FOLDS
    
    # Calculate fold SMAPE
    val_pred_price = np.expm1(oof_preds[val_idx])
    val_actual_price = np.expm1(y_train_log[val_idx])
    
    fold_smape = np.mean(
        2 * np.abs(val_pred_price - val_actual_price) / 
        (np.abs(val_actual_price) + np.abs(val_pred_price) + 1e-8)
    ) * 100
    
    fold_scores.append(fold_smape)
    print(f"  📈 Fold {fold} SMAPE: {fold_smape:.4f}%")
    
    del model
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

# ===================================================================
# FINAL EVALUATION
# ===================================================================
print("\n[5/5] Final evaluation and submission...")

oof_prices = np.expm1(oof_preds)
actual_prices = df_train['price'].values

overall_smape = np.mean(
    2 * np.abs(oof_prices - actual_prices) / 
    (np.abs(actual_prices) + np.abs(oof_prices) + 1e-8)
) * 100

print("\n" + "="*70)
print(f"📊 CROSS-VALIDATION RESULTS")
print("="*70)
for i, score in enumerate(fold_scores, 1):
    print(f"  Fold {i}: {score:.4f}%")
print(f"\n  Mean: {np.mean(fold_scores):.4f}%")
print(f"  Std:  {np.std(fold_scores):.4f}%")
print("\n" + "="*70)
print(f"🎯 FINAL OOF SMAPE: {overall_smape:.4f}%")
print("="*70)

# ===================================================================
# CREATE SUBMISSION
# ===================================================================
final_predictions = np.expm1(test_preds)
final_predictions = np.clip(final_predictions, 0.01, None)

submission = pd.DataFrame({
    'sample_id': df_test['sample_id'],
    'price': final_predictions
})

submission.to_csv('enhanced_mlp_fusion_submission.csv', index=False)

print("\n✅ Submission saved: enhanced_mlp_fusion_submission.csv")
print("\n📋 Prediction statistics:")
print(f"  Min:    ${final_predictions.min():.2f}")
print(f"  Max:    ${final_predictions.max():.2f}")
print(f"  Mean:   ${final_predictions.mean():.2f}")
print(f"  Median: ${np.median(final_predictions):.2f}")
print("\n" + "="*70)

🚀 ENHANCED MLP FUSION: Maximum Feature Extraction

[1/5] Loading data and embeddings...
✓ Loaded embeddings

[2/5] Engineering advanced features...
  Extracting advanced features...
  Extracting advanced features...
✓ Enhanced features: 165 dimensions

[3/5] Applying robust scaling...
✓ Scaling complete

[4/5] Training with K-Fold CV...

──────────────────────────────────────────────────────────────────────
📊 FOLD 1/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25088, val_loss=0.33416
    Epoch 20: train_loss=0.22641, val_loss=0.33652
    Epoch 30: train_loss=0.16866, val_loss=0.32717
    Epoch 40: train_loss=0.13642, val_loss=0.32135
    Epoch 50: train_loss=0.15529, val_loss=0.32713
    Epoch 60: train_loss=0.13190, val_loss=0.32637
    Early stopping at epoch 62


  📈 Fold 1 SMAPE: 47.8323%

──────────────────────────────────────────────────────────────────────
📊 FOLD 2/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25117, val_loss=0.32323
    Epoch 20: train_loss=0.22577, val_loss=0.33532
    Epoch 30: train_loss=0.16981, val_loss=0.32053
    Epoch 40: train_loss=0.13569, val_loss=0.31742
    Epoch 50: train_loss=0.15794, val_loss=0.32438
    Early stopping at epoch 56


  📈 Fold 2 SMAPE: 47.8493%

──────────────────────────────────────────────────────────────────────
📊 FOLD 3/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25196, val_loss=0.33515
    Epoch 20: train_loss=0.22621, val_loss=0.33178
    Epoch 30: train_loss=0.17042, val_loss=0.32637
    Epoch 40: train_loss=0.13632, val_loss=0.32140
    Epoch 50: train_loss=0.15751, val_loss=0.33357
    Early stopping at epoch 58


  📈 Fold 3 SMAPE: 47.9544%

──────────────────────────────────────────────────────────────────────
📊 FOLD 4/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25147, val_loss=0.32622
    Epoch 20: train_loss=0.22674, val_loss=0.31911
    Epoch 30: train_loss=0.17071, val_loss=0.32221
    Epoch 40: train_loss=0.13659, val_loss=0.31804
    Epoch 50: train_loss=0.15780, val_loss=0.32506
    Early stopping at epoch 59


  📈 Fold 4 SMAPE: 47.4705%

──────────────────────────────────────────────────────────────────────
📊 FOLD 5/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25466, val_loss=0.31553
    Epoch 20: train_loss=0.22879, val_loss=0.32339
    Epoch 30: train_loss=0.17104, val_loss=0.31211
    Epoch 40: train_loss=0.13809, val_loss=0.30788
    Epoch 50: train_loss=0.15855, val_loss=0.31612
    Epoch 60: train_loss=0.13377, val_loss=0.31208
    Early stopping at epoch 64


  📈 Fold 5 SMAPE: 46.2245%

──────────────────────────────────────────────────────────────────────
📊 FOLD 6/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25322, val_loss=0.33101
    Epoch 20: train_loss=0.22744, val_loss=0.34008
    Epoch 30: train_loss=0.17083, val_loss=0.32817
    Epoch 40: train_loss=0.13801, val_loss=0.32253
    Epoch 50: train_loss=0.15747, val_loss=0.32311
    Epoch 60: train_loss=0.13338, val_loss=0.32580
    Epoch 70: train_loss=0.11240, val_loss=0.32114
    Epoch 80: train_loss=0.09699, val_loss=0.31858
    Epoch 90: train_loss=0.08649, val_loss=0.31460
    Epoch 100: train_loss=0.08139, val_loss=0.31357
    Epoch 110: train_loss=0.10912, val_loss=0.32766
    Early stopping at epoch 113


  📈 Fold 6 SMAPE: 47.0406%

──────────────────────────────────────────────────────────────────────
📊 FOLD 7/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.25069, val_loss=0.32917
    Epoch 20: train_loss=0.22715, val_loss=0.32947
    Epoch 30: train_loss=0.16967, val_loss=0.31947
    Epoch 40: train_loss=0.13660, val_loss=0.31726
    Epoch 50: train_loss=0.15748, val_loss=0.31955
    Epoch 60: train_loss=0.13309, val_loss=0.31878
    Early stopping at epoch 64


  📈 Fold 7 SMAPE: 47.0506%

[5/5] Final evaluation and submission...

📊 CROSS-VALIDATION RESULTS
  Fold 1: 47.8323%
  Fold 2: 47.8493%
  Fold 3: 47.9544%
  Fold 4: 47.4705%
  Fold 5: 46.2245%
  Fold 6: 47.0406%
  Fold 7: 47.0506%

  Mean: 47.3460%
  Std:  0.5749%

🎯 FINAL OOF SMAPE: 47.3461%

✅ Submission saved: enhanced_mlp_fusion_submission.csv

📋 Prediction statistics:
  Min:    $0.26
  Max:    $365.00
  Mean:   $20.45
  Median: $13.26



claude ultimate 3


In [12]:
# ===================================================================
# ENHANCED MLP FUSION WITH ADVANCED FEATURE ENGINEERING
# Target: SMAPE < 45% with Maximum Data Extraction
# ===================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from tqdm import tqdm
import gc
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 ENHANCED MLP FUSION: Maximum Feature Extraction")
print("="*70)

# ===================================================================
# ADVANCED FEATURE ENGINEERING
# ===================================================================
def extract_advanced_features(df, is_train=True):
    """Extract rich features from catalog_content and other columns"""
    print(f"  Extracting advanced features...")
    
    features = {}
    
    # Text length features
    features['title_len'] = df['catalog_content'].str.len()
    features['word_count'] = df['catalog_content'].str.split().str.len()
    features['avg_word_len'] = features['title_len'] / (features['word_count'] + 1)
    
    # Brand features (if present)
    if 'brand' in df.columns:
        # Brand frequency encoding
        brand_freq = df['brand'].value_counts()
        features['brand_freq'] = df['brand'].map(brand_freq).fillna(0)
        
        # Brand mean price (only for train)
        if is_train and 'price' in df.columns:
            brand_mean = df.groupby('brand')['price'].mean()
            features['brand_mean_price'] = df['brand'].map(brand_mean).fillna(df['price'].median())
    
    # Quantity features
    if 'quantity' in df.columns:
        features['quantity'] = df['quantity'].fillna(1)
        features['log_quantity'] = np.log1p(features['quantity'])
        features['quantity_squared'] = features['quantity'] ** 2
    
    # Text pattern features
    features['has_digits'] = df['catalog_content'].str.contains(r'\d', regex=True).astype(int)
    features['has_special_chars'] = df['catalog_content'].str.contains(r'[^a-zA-Z0-9\s]', regex=True).astype(int)
    features['uppercase_ratio'] = df['catalog_content'].str.count(r'[A-Z]') / (features['title_len'] + 1)
    
    # Extract numeric values from text
    features['num_numbers'] = df['catalog_content'].str.findall(r'\d+').str.len().fillna(0)
    
    # Check for common price indicators
    price_keywords = ['premium', 'luxury', 'pro', 'plus', 'max', 'ultra', 'deluxe']
    budget_keywords = ['basic', 'mini', 'lite', 'eco', 'value']
    
    features['has_premium_word'] = df['catalog_content'].str.lower().str.contains('|'.join(price_keywords)).astype(int)
    features['has_budget_word'] = df['catalog_content'].str.lower().str.contains('|'.join(budget_keywords)).astype(int)
    
    return pd.DataFrame(features)

# ===================================================================
# ENHANCED MLP ARCHITECTURE WITH RESIDUAL CONNECTIONS
# ===================================================================
class EnhancedMultimodalFusionMLP(nn.Module):
    """
    Enhanced fusion with:
    - Residual connections
    - Better attention mechanism
    - Gated fusion
    - More sophisticated architecture
    """
    def __init__(self, text_dim, image_dim, other_dim, hidden_dim=1024, dropout=0.3):
        super().__init__()
        
        # Individual encoders with residual capability
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.other_encoder = nn.Sequential(
            nn.Linear(other_dim, 192),
            nn.LayerNorm(192),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(192, 192),
            nn.LayerNorm(192),
            nn.GELU()
        )
        
        # Cross-attention between modalities
        self.text_to_image_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        self.image_to_text_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        
        # Gated fusion mechanism
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 192, hidden_dim * 2),  # Changed from 128 to 192
            nn.Sigmoid()
        )
        
        # Main fusion network with residual connections
        fusion_input_dim = hidden_dim * 2 + 192  # Changed from 128 to 192
        self.fusion1 = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.fusion2 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.7)
        )
        
        self.fusion3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        
        self.output = nn.Linear(hidden_dim // 2, 1)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, text_emb, image_emb, other_emb):
        # Encode each modality
        text_enc = self.text_encoder(text_emb)
        image_enc = self.image_encoder(image_emb)
        other_enc = self.other_encoder(other_emb)
        
        # Bidirectional cross-attention
        text_att, _ = self.text_to_image_attn(
            text_enc.unsqueeze(1), 
            image_enc.unsqueeze(1), 
            image_enc.unsqueeze(1)
        )
        text_att = text_att.squeeze(1)
        
        image_att, _ = self.image_to_text_attn(
            image_enc.unsqueeze(1), 
            text_enc.unsqueeze(1), 
            text_enc.unsqueeze(1)
        )
        image_att = image_att.squeeze(1)
        
        # Combine with residuals
        text_combined = text_enc + text_att
        image_combined = image_enc + image_att
        
        # Concatenate all features
        fused = torch.cat([text_combined, image_combined, other_enc], dim=1)
        
        # Gated fusion
        gate_values = self.gate(fused)
        
        # Apply fusion layers with residuals
        x = self.fusion1(fused)
        x = x * gate_values[:, :x.size(1)]  # Apply gating
        
        x = self.fusion2(x)
        x = self.fusion3(x)
        output = self.output(x)
        
        return output

# ===================================================================
# CUSTOM LOSS FUNCTION - SMAPE-inspired
# ===================================================================
def smape_loss(pred, target, epsilon=1e-3):
    """SMAPE-inspired loss that directly optimizes the evaluation metric"""
    # Work in log space to match our target
    pred_exp = torch.exp(pred)
    target_exp = torch.exp(target)
    
    numerator = torch.abs(pred_exp - target_exp)
    denominator = (torch.abs(target_exp) + torch.abs(pred_exp)) / 2.0 + epsilon
    
    return torch.mean(numerator / denominator)

def combined_loss(pred, target, alpha=0.6):
    """Combine SMAPE loss with Huber loss for stability"""
    smape = smape_loss(pred, target)
    huber = nn.SmoothL1Loss()(pred, target)
    return alpha * smape + (1 - alpha) * huber

# ===================================================================
# TRAINING FUNCTION WITH ADVANCED TECHNIQUES
# ===================================================================
def train_enhanced_mlp(X_text_tr, X_image_tr, X_other_tr, y_tr, 
                       X_text_val, X_image_val, X_other_val, y_val,
                       epochs=250, batch_size=192, lr=1.5e-4):
    
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"  Using device: {device}")
    
    text_dim = X_text_tr.shape[1]
    image_dim = X_image_tr.shape[1]
    other_dim = X_other_tr.shape[1]
    
    model = EnhancedMultimodalFusionMLP(text_dim, image_dim, other_dim).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-5, betas=(0.9, 0.999))
    
    # Cosine annealing with warm restarts
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=15, T_mult=2, eta_min=1e-6
    )
    
    # Create datasets
    train_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_text_tr),
        torch.FloatTensor(X_image_tr),
        torch.FloatTensor(X_other_tr),
        torch.FloatTensor(y_tr).unsqueeze(1)
    )
    
    val_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_text_val),
        torch.FloatTensor(X_image_val),
        torch.FloatTensor(X_other_val),
        torch.FloatTensor(y_val).unsqueeze(1)
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    best_val_loss = float('inf')
    patience = 25  # Increased patience for more training
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        
        for text_b, image_b, other_b, y_b in train_loader:
            text_b = text_b.to(device)
            image_b = image_b.to(device)
            other_b = other_b.to(device)
            y_b = y_b.to(device)
            
            optimizer.zero_grad()
            output = model(text_b, image_b, other_b)
            loss = combined_loss(output, y_b)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for text_b, image_b, other_b, y_b in val_loader:
                text_b = text_b.to(device)
                image_b = image_b.to(device)
                other_b = other_b.to(device)
                y_b = y_b.to(device)
                
                output = model(text_b, image_b, other_b)
                val_loss += combined_loss(output, y_b).item()
        
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        scheduler.step()
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1}")
            break
        
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}: train_loss={train_loss:.5f}, val_loss={val_loss:.5f}")
    
    model.load_state_dict(best_model_state)
    return model

# ===================================================================
# PREDICTION FUNCTION
# ===================================================================
def predict_enhanced(model, X_text, X_image, X_other, batch_size=512):
    device = next(model.parameters()).device
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(X_text), batch_size), desc="Predicting", leave=False):
            end_idx = min(i + batch_size, len(X_text))
            
            text_b = torch.FloatTensor(X_text[i:end_idx]).to(device)
            image_b = torch.FloatTensor(X_image[i:end_idx]).to(device)
            other_b = torch.FloatTensor(X_other[i:end_idx]).to(device)
            
            output = model(text_b, image_b, other_b)
            predictions.append(output.cpu().numpy())
    
    return np.vstack(predictions).flatten()

# ===================================================================
# MAIN EXECUTION
# ===================================================================
print("\n[1/5] Loading data and embeddings...")
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Load embeddings
X_train_full = np.load("final_X_train_medium_with_brand.npy", allow_pickle=False)
X_test_full = np.load("final_X_test_medium_with_brand.npy", allow_pickle=False)

# Define dimensions
text_dim = 384
image_dim = 512

# Slice embeddings
train_text = X_train_full[:, :text_dim]
train_image = X_train_full[:, text_dim:text_dim+image_dim]
train_other_base = X_train_full[:, text_dim+image_dim:]

test_text = X_test_full[:, :text_dim]
test_image = X_test_full[:, text_dim:text_dim+image_dim]
test_other_base = X_test_full[:, text_dim+image_dim:]

print(f"✓ Loaded embeddings")
del X_train_full, X_test_full
gc.collect()

# ===================================================================
# ADVANCED FEATURE ENGINEERING
# ===================================================================
print("\n[2/5] Engineering advanced features...")
train_extra_features = extract_advanced_features(df_train, is_train=True)
test_extra_features = extract_advanced_features(df_test, is_train=False)

# Combine with existing other features
train_other = np.hstack([train_other_base, train_extra_features.values])
test_other = np.hstack([test_other_base, test_extra_features.values])

print(f"✓ Enhanced features: {train_other.shape[1]} dimensions")
del train_other_base, test_other_base
gc.collect()

# Target transformation
y_train_log = np.log1p(df_train['price'].values)

# ===================================================================
# ADVANCED SCALING
# ===================================================================
print("\n[3/5] Applying robust scaling...")

# Use QuantileTransformer for embeddings (more robust to outliers)
text_scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal')
image_scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal')
other_scaler = RobustScaler()

train_text_scaled = text_scaler.fit_transform(train_text)
test_text_scaled = text_scaler.transform(test_text)

train_image_scaled = image_scaler.fit_transform(train_image)
test_image_scaled = image_scaler.transform(test_image)

train_other_scaled = other_scaler.fit_transform(train_other)
test_other_scaled = other_scaler.transform(test_other)

print("✓ Scaling complete")
del train_text, train_image, train_other, test_text, test_image, test_other
gc.collect()

# ===================================================================
# K-FOLD CROSS-VALIDATION WITH STRATIFICATION
# ===================================================================
print("\n[4/5] Training with K-Fold CV...")

N_FOLDS = 7  # Increased from 5 to 7 for better generalization
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_text_scaled))
test_preds = np.zeros(len(test_text_scaled))

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_text_scaled), 1):
    print(f"\n{'─'*70}")
    print(f"📊 FOLD {fold}/{N_FOLDS}")
    print(f"{'─'*70}")
    
    model = train_enhanced_mlp(
        train_text_scaled[train_idx], 
        train_image_scaled[train_idx], 
        train_other_scaled[train_idx], 
        y_train_log[train_idx],
        train_text_scaled[val_idx], 
        train_image_scaled[val_idx], 
        train_other_scaled[val_idx], 
        y_train_log[val_idx]
    )
    
    # OOF predictions
    oof_preds[val_idx] = predict_enhanced(
        model, 
        train_text_scaled[val_idx], 
        train_image_scaled[val_idx], 
        train_other_scaled[val_idx]
    )
    
    # Test predictions
    fold_test_preds = predict_enhanced(
        model, 
        test_text_scaled, 
        test_image_scaled, 
        test_other_scaled
    )
    test_preds += fold_test_preds / N_FOLDS
    
    # Calculate fold SMAPE
    val_pred_price = np.expm1(oof_preds[val_idx])
    val_actual_price = np.expm1(y_train_log[val_idx])
    
    fold_smape = np.mean(
        2 * np.abs(val_pred_price - val_actual_price) / 
        (np.abs(val_actual_price) + np.abs(val_pred_price) + 1e-8)
    ) * 100
    
    fold_scores.append(fold_smape)
    print(f"  📈 Fold {fold} SMAPE: {fold_smape:.4f}%")
    
    del model
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

# ===================================================================
# FINAL EVALUATION
# ===================================================================
print("\n[5/5] Final evaluation and submission...")

oof_prices = np.expm1(oof_preds)
actual_prices = df_train['price'].values

overall_smape = np.mean(
    2 * np.abs(oof_prices - actual_prices) / 
    (np.abs(actual_prices) + np.abs(oof_prices) + 1e-8)
) * 100

print("\n" + "="*70)
print(f"📊 CROSS-VALIDATION RESULTS")
print("="*70)
for i, score in enumerate(fold_scores, 1):
    print(f"  Fold {i}: {score:.4f}%")
print(f"\n  Mean: {np.mean(fold_scores):.4f}%")
print(f"  Std:  {np.std(fold_scores):.4f}%")
print("\n" + "="*70)
print(f"🎯 FINAL OOF SMAPE: {overall_smape:.4f}%")
print("="*70)

# ===================================================================
# CREATE SUBMISSION
# ===================================================================
final_predictions = np.expm1(test_preds)
final_predictions = np.clip(final_predictions, 0.01, None)

submission = pd.DataFrame({
    'sample_id': df_test['sample_id'],
    'price': final_predictions
})

submission.to_csv('enhanced_mlp_fusion_submission.csv', index=False)

print("\n✅ Submission saved: enhanced_mlp_fusion_submission.csv")
print("\n📋 Prediction statistics:")
print(f"  Min:    ${final_predictions.min():.2f}")
print(f"  Max:    ${final_predictions.max():.2f}")
print(f"  Mean:   ${final_predictions.mean():.2f}")
print(f"  Median: ${np.median(final_predictions):.2f}")
print("\n" + "="*70)

🚀 ENHANCED MLP FUSION: Maximum Feature Extraction

[1/5] Loading data and embeddings...
✓ Loaded embeddings

[2/5] Engineering advanced features...
  Extracting advanced features...
  Extracting advanced features...
✓ Enhanced features: 165 dimensions

[3/5] Applying robust scaling...
✓ Scaling complete

[4/5] Training with K-Fold CV...

──────────────────────────────────────────────────────────────────────
📊 FOLD 1/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.30413, val_loss=0.36193
    Epoch 20: train_loss=0.27694, val_loss=0.36145
    Epoch 30: train_loss=0.21626, val_loss=0.34832
    Epoch 40: train_loss=0.18167, val_loss=0.34659
    Epoch 50: train_loss=0.19529, val_loss=0.35304
    Epoch 60: train_loss=0.16814, val_loss=0.35103
    Epoch 70: train_loss=0.14418, val_loss=0.34519
    Epoch 80: train_loss=0.12544, val_loss=0.34139
    Epoch 90: train_loss=0.11394, val_loss=0.33946
    Epoch 100: train_loss=0.

  📈 Fold 1 SMAPE: 47.1083%

──────────────────────────────────────────────────────────────────────
📊 FOLD 2/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.30823, val_loss=0.35781
    Epoch 20: train_loss=0.27977, val_loss=0.34860
    Epoch 30: train_loss=0.21856, val_loss=0.34268
    Epoch 40: train_loss=0.18251, val_loss=0.33852
    Epoch 50: train_loss=0.19565, val_loss=0.34271
    Epoch 60: train_loss=0.16900, val_loss=0.33864
    Epoch 70: train_loss=0.14464, val_loss=0.34018
    Epoch 80: train_loss=0.12655, val_loss=0.33538
    Epoch 90: train_loss=0.11436, val_loss=0.33216
    Epoch 100: train_loss=0.10929, val_loss=0.33180
    Epoch 110: train_loss=0.13147, val_loss=0.33781
    Early stopping at epoch 118


  📈 Fold 2 SMAPE: 46.6410%

──────────────────────────────────────────────────────────────────────
📊 FOLD 3/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.30651, val_loss=0.35723
    Epoch 20: train_loss=0.27693, val_loss=0.35604
    Epoch 30: train_loss=0.21798, val_loss=0.34763
    Epoch 40: train_loss=0.18245, val_loss=0.34037
    Epoch 50: train_loss=0.19689, val_loss=0.35819
    Epoch 60: train_loss=0.16879, val_loss=0.34326
    Epoch 70: train_loss=0.14482, val_loss=0.34626
    Epoch 80: train_loss=0.12568, val_loss=0.34112
    Epoch 90: train_loss=0.11395, val_loss=0.33861
    Epoch 100: train_loss=0.10878, val_loss=0.33543
    Epoch 110: train_loss=0.13217, val_loss=0.34557
    Epoch 120: train_loss=0.12168, val_loss=0.34236
    Early stopping at epoch 129


  📈 Fold 3 SMAPE: 46.4797%

──────────────────────────────────────────────────────────────────────
📊 FOLD 4/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.30669, val_loss=0.34968
    Epoch 20: train_loss=0.27783, val_loss=0.35257
    Epoch 30: train_loss=0.21812, val_loss=0.34560
    Epoch 40: train_loss=0.18397, val_loss=0.33681
    Epoch 50: train_loss=0.19728, val_loss=0.33962
    Epoch 60: train_loss=0.16873, val_loss=0.34092
    Early stopping at epoch 67


  📈 Fold 4 SMAPE: 46.7282%

──────────────────────────────────────────────────────────────────────
📊 FOLD 5/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.30757, val_loss=0.34275
    Epoch 20: train_loss=0.27818, val_loss=0.34034
    Epoch 30: train_loss=0.21716, val_loss=0.33484
    Epoch 40: train_loss=0.18279, val_loss=0.33329
    Epoch 50: train_loss=0.19700, val_loss=0.34481
    Epoch 60: train_loss=0.16938, val_loss=0.34229
    Early stopping at epoch 67


  📈 Fold 5 SMAPE: 46.6525%

──────────────────────────────────────────────────────────────────────
📊 FOLD 6/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.30650, val_loss=0.35846
    Epoch 20: train_loss=0.27908, val_loss=0.35206
    Epoch 30: train_loss=0.21759, val_loss=0.34442
    Epoch 40: train_loss=0.18178, val_loss=0.34314
    Epoch 50: train_loss=0.19757, val_loss=0.35353
    Epoch 60: train_loss=0.16923, val_loss=0.34294
    Epoch 70: train_loss=0.14469, val_loss=0.34040
    Epoch 80: train_loss=0.12548, val_loss=0.33880
    Epoch 90: train_loss=0.11384, val_loss=0.33696
    Epoch 100: train_loss=0.10921, val_loss=0.33646
    Epoch 110: train_loss=0.13154, val_loss=0.34605
    Early stopping at epoch 112


  📈 Fold 6 SMAPE: 47.3218%

──────────────────────────────────────────────────────────────────────
📊 FOLD 7/7
──────────────────────────────────────────────────────────────────────
  Using device: mps
    Epoch 10: train_loss=0.30595, val_loss=0.35471
    Epoch 20: train_loss=0.27640, val_loss=0.35303
    Epoch 30: train_loss=0.21652, val_loss=0.34087
    Epoch 40: train_loss=0.18206, val_loss=0.33679
    Epoch 50: train_loss=0.19644, val_loss=0.34989
    Epoch 60: train_loss=0.16666, val_loss=0.33945
    Epoch 70: train_loss=0.14449, val_loss=0.33964
    Epoch 80: train_loss=0.12609, val_loss=0.33451
    Epoch 90: train_loss=0.11402, val_loss=0.33062
    Epoch 100: train_loss=0.10861, val_loss=0.32905
    Epoch 110: train_loss=0.13171, val_loss=0.33810
    Epoch 120: train_loss=0.12264, val_loss=0.33830
    Early stopping at epoch 124


  📈 Fold 7 SMAPE: 46.1620%

[5/5] Final evaluation and submission...

📊 CROSS-VALIDATION RESULTS
  Fold 1: 47.1083%
  Fold 2: 46.6410%
  Fold 3: 46.4797%
  Fold 4: 46.7282%
  Fold 5: 46.6525%
  Fold 6: 47.3218%
  Fold 7: 46.1620%

  Mean: 46.7276%
  Std:  0.3571%

🎯 FINAL OOF SMAPE: 46.7277%

✅ Submission saved: enhanced_mlp_fusion_submission.csv

📋 Prediction statistics:
  Min:    $0.30
  Max:    $375.96
  Mean:   $20.56
  Median: $13.39



In [13]:
# ===================================================================
# OPTIMIZED FAST MLP FUSION - Target: <1 hour, SMAPE < 40%
# Key optimizations: Smaller model, fewer folds, faster convergence
# ===================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("⚡ FAST MLP FUSION: Optimized for Speed & Performance")
print("="*70)

# ===================================================================
# STREAMLINED FEATURE ENGINEERING
# ===================================================================
def extract_fast_features(df, brand_stats=None, is_train=True):
    """Efficient feature extraction with minimal computation"""
    features = {}
    
    # Basic text features (vectorized)
    features['title_len'] = df['catalog_content'].str.len()
    features['word_count'] = df['catalog_content'].str.split().str.len()
    features['avg_word_len'] = features['title_len'] / (features['word_count'] + 1)
    
    # Brand features
    if 'brand' in df.columns:
        if is_train:
            brand_freq = df['brand'].value_counts()
            brand_mean = df.groupby('brand')['price'].mean()
            brand_std = df.groupby('brand')['price'].std().fillna(0)
            brand_stats = {'freq': brand_freq, 'mean': brand_mean, 'std': brand_std}
        
        if brand_stats:
            features['brand_freq'] = df['brand'].map(brand_stats['freq']).fillna(0)
            features['brand_mean_price'] = df['brand'].map(brand_stats['mean']).fillna(
                brand_stats['mean'].median() if is_train else 0
            )
            features['brand_std_price'] = df['brand'].map(brand_stats['std']).fillna(0)
    
    # Quantity features
    if 'quantity' in df.columns:
        features['quantity'] = df['quantity'].fillna(1)
        features['log_quantity'] = np.log1p(features['quantity'])
    
    # Fast text patterns
    features['has_digits'] = df['catalog_content'].str.contains(r'\d', regex=True).astype(int)
    features['num_numbers'] = df['catalog_content'].str.findall(r'\d+').str.len().fillna(0)
    
    # Price indicators (combined regex for speed)
    price_pattern = 'premium|luxury|pro|plus|max|ultra|deluxe'
    features['has_premium_word'] = df['catalog_content'].str.lower().str.contains(price_pattern).astype(int)
    
    return pd.DataFrame(features), brand_stats if is_train else None

# ===================================================================
# EFFICIENT MLP ARCHITECTURE
# ===================================================================
class FastMultimodalMLP(nn.Module):
    """
    Streamlined architecture:
    - Smaller hidden dimensions
    - Fewer layers
    - Efficient attention
    - Faster training
    """
    def __init__(self, text_dim, image_dim, other_dim, hidden_dim=512, dropout=0.25):
        super().__init__()
        
        # Lightweight encoders
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        self.other_encoder = nn.Sequential(
            nn.Linear(other_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        
        # Single efficient cross-attention
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        
        # Compact fusion network
        fusion_dim = hidden_dim * 2 + 128
        
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, text_emb, image_emb, other_emb):
        # Encode modalities
        text_enc = self.text_encoder(text_emb)
        image_enc = self.image_encoder(image_emb)
        other_enc = self.other_encoder(other_emb)
        
        # Efficient bidirectional attention
        text_att, _ = self.cross_attn(
            text_enc.unsqueeze(1), 
            image_enc.unsqueeze(1), 
            image_enc.unsqueeze(1)
        )
        text_combined = text_enc + text_att.squeeze(1)
        
        image_att, _ = self.cross_attn(
            image_enc.unsqueeze(1), 
            text_enc.unsqueeze(1), 
            text_enc.unsqueeze(1)
        )
        image_combined = image_enc + image_att.squeeze(1)
        
        # Fuse and predict
        fused = torch.cat([text_combined, image_combined, other_enc], dim=1)
        output = self.fusion(fused)
        
        return output

# ===================================================================
# FAST TRAINING WITH AGGRESSIVE CONVERGENCE
# ===================================================================
def train_fast(X_text_tr, X_image_tr, X_other_tr, y_tr, 
               X_text_val, X_image_val, X_other_val, y_val,
               epochs=80, batch_size=256, lr=3e-4):
    
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    
    model = FastMultimodalMLP(
        X_text_tr.shape[1], 
        X_image_tr.shape[1], 
        X_other_tr.shape[1]
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4, betas=(0.9, 0.999))
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, 
        steps_per_epoch=len(X_text_tr) // batch_size + 1,
        pct_start=0.1
    )
    
    # Create datasets
    train_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_text_tr),
        torch.FloatTensor(X_image_tr),
        torch.FloatTensor(X_other_tr),
        torch.FloatTensor(y_tr).unsqueeze(1)
    )
    
    val_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_text_val),
        torch.FloatTensor(X_image_val),
        torch.FloatTensor(X_other_val),
        torch.FloatTensor(y_val).unsqueeze(1)
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size*2, shuffle=False, 
                           num_workers=0, pin_memory=True)
    
    criterion = nn.SmoothL1Loss()
    best_val_loss = float('inf')
    patience = 12
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        
        for text_b, image_b, other_b, y_b in train_loader:
            text_b, image_b, other_b, y_b = (
                text_b.to(device), image_b.to(device), 
                other_b.to(device), y_b.to(device)
            )
            
            optimizer.zero_grad()
            output = model(text_b, image_b, other_b)
            loss = criterion(output, y_b)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for text_b, image_b, other_b, y_b in val_loader:
                text_b, image_b, other_b, y_b = (
                    text_b.to(device), image_b.to(device), 
                    other_b.to(device), y_b.to(device)
                )
                output = model(text_b, image_b, other_b)
                val_loss += criterion(output, y_b).item()
        
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"    Early stop at epoch {epoch+1}")
            break
        
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}: train={train_loss:.5f}, val={val_loss:.5f}")
    
    model.load_state_dict(best_model_state)
    return model

# ===================================================================
# FAST PREDICTION
# ===================================================================
def predict_fast(model, X_text, X_image, X_other, batch_size=1024):
    device = next(model.parameters()).device
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for i in range(0, len(X_text), batch_size):
            end_idx = min(i + batch_size, len(X_text))
            
            text_b = torch.FloatTensor(X_text[i:end_idx]).to(device)
            image_b = torch.FloatTensor(X_image[i:end_idx]).to(device)
            other_b = torch.FloatTensor(X_other[i:end_idx]).to(device)
            
            output = model(text_b, image_b, other_b)
            predictions.append(output.cpu().numpy())
    
    return np.vstack(predictions).flatten()

# ===================================================================
# MAIN EXECUTION
# ===================================================================
print("\n[1/4] Loading data...")
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

X_train_full = np.load("final_X_train_medium_with_brand.npy", allow_pickle=False)
X_test_full = np.load("final_X_test_medium_with_brand.npy", allow_pickle=False)

text_dim, image_dim = 384, 512

train_text = X_train_full[:, :text_dim]
train_image = X_train_full[:, text_dim:text_dim+image_dim]
train_other_base = X_train_full[:, text_dim+image_dim:]

test_text = X_test_full[:, :text_dim]
test_image = X_test_full[:, text_dim:text_dim+image_dim]
test_other_base = X_test_full[:, text_dim+image_dim:]

del X_train_full, X_test_full
gc.collect()

# ===================================================================
# FEATURE ENGINEERING
# ===================================================================
print("\n[2/4] Feature engineering...")
train_extra, brand_stats = extract_fast_features(df_train, is_train=True)
test_extra, _ = extract_fast_features(df_test, brand_stats, is_train=False)

train_other = np.hstack([train_other_base, train_extra.values])
test_other = np.hstack([test_other_base, test_extra.values])

print(f"✓ Features: {train_other.shape[1]} dimensions")
del train_other_base, test_other_base
gc.collect()

y_train_log = np.log1p(df_train['price'].values)

# ===================================================================
# FAST SCALING
# ===================================================================
print("\n[3/4] Scaling...")
scaler = RobustScaler()

train_text_scaled = scaler.fit_transform(train_text)
test_text_scaled = scaler.transform(test_text)

train_image_scaled = scaler.fit_transform(train_image)
test_image_scaled = scaler.transform(test_image)

train_other_scaled = scaler.fit_transform(train_other)
test_other_scaled = scaler.transform(test_other)

del train_text, train_image, train_other, test_text, test_image, test_other
gc.collect()

# ===================================================================
# FAST K-FOLD (4 FOLDS)
# ===================================================================
print("\n[4/4] Training (4-Fold CV)...")

N_FOLDS = 4
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_text_scaled))
test_preds = np.zeros(len(test_text_scaled))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_text_scaled), 1):
    print(f"\n{'─'*70}")
    print(f"📊 FOLD {fold}/{N_FOLDS}")
    print(f"{'─'*70}")
    
    model = train_fast(
        train_text_scaled[train_idx], 
        train_image_scaled[train_idx], 
        train_other_scaled[train_idx], 
        y_train_log[train_idx],
        train_text_scaled[val_idx], 
        train_image_scaled[val_idx], 
        train_other_scaled[val_idx], 
        y_train_log[val_idx]
    )
    
    # OOF predictions
    oof_preds[val_idx] = predict_fast(
        model, 
        train_text_scaled[val_idx], 
        train_image_scaled[val_idx], 
        train_other_scaled[val_idx]
    )
    
    # Test predictions
    fold_test_preds = predict_fast(
        model, test_text_scaled, test_image_scaled, test_other_scaled
    )
    test_preds += fold_test_preds / N_FOLDS
    
    # Calculate SMAPE
    val_pred_price = np.expm1(oof_preds[val_idx])
    val_actual_price = np.expm1(y_train_log[val_idx])
    
    fold_smape = np.mean(
        2 * np.abs(val_pred_price - val_actual_price) / 
        (np.abs(val_actual_price) + np.abs(val_pred_price) + 1e-8)
    ) * 100
    
    fold_scores.append(fold_smape)
    print(f"  📈 Fold {fold} SMAPE: {fold_smape:.4f}%")
    
    del model
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

# ===================================================================
# FINAL RESULTS
# ===================================================================
oof_prices = np.expm1(oof_preds)
actual_prices = df_train['price'].values

overall_smape = np.mean(
    2 * np.abs(oof_prices - actual_prices) / 
    (np.abs(actual_prices) + np.abs(oof_prices) + 1e-8)
) * 100

print("\n" + "="*70)
print(f"📊 FINAL RESULTS")
print("="*70)
for i, score in enumerate(fold_scores, 1):
    print(f"  Fold {i}: {score:.4f}%")
print(f"\n  Mean: {np.mean(fold_scores):.4f}%")
print(f"  Std:  {np.std(fold_scores):.4f}%")
print(f"\n🎯 OOF SMAPE: {overall_smape:.4f}%")
print("="*70)

# ===================================================================
# SUBMISSION
# ===================================================================
final_predictions = np.expm1(test_preds)
final_predictions = np.clip(final_predictions, 0.01, None)

submission = pd.DataFrame({
    'sample_id': df_test['sample_id'],
    'price': final_predictions
})

submission.to_csv('fast_mlp_fusion_submission.csv', index=False)

print("\n✅ Saved: fast_mlp_fusion_submission.csv")
print(f"\n📋 Stats:")
print(f"  Min:    ${final_predictions.min():.2f}")
print(f"  Max:    ${final_predictions.max():.2f}")
print(f"  Mean:   ${final_predictions.mean():.2f}")
print(f"  Median: ${np.median(final_predictions):.2f}")
print("\n" + "="*70)

⚡ FAST MLP FUSION: Optimized for Speed & Performance

[1/4] Loading data...

[2/4] Feature engineering...
✓ Features: 162 dimensions

[3/4] Scaling...

[4/4] Training (4-Fold CV)...

──────────────────────────────────────────────────────────────────────
📊 FOLD 1/4
──────────────────────────────────────────────────────────────────────
    Epoch 10: train=0.18182, val=0.21834
    Epoch 20: train=0.11376, val=0.21209
    Early stop at epoch 26
  📈 Fold 1 SMAPE: 50.5884%

──────────────────────────────────────────────────────────────────────
📊 FOLD 2/4
──────────────────────────────────────────────────────────────────────
    Epoch 10: train=0.18081, val=0.21482
    Epoch 20: train=0.11367, val=0.20936
    Epoch 30: train=0.08264, val=0.21520
    Early stop at epoch 38
  📈 Fold 2 SMAPE: 49.9212%

──────────────────────────────────────────────────────────────────────
📊 FOLD 3/4
──────────────────────────────────────────────────────────────────────
    Epoch 10: train=0.18310, val=0.20624
  